# PEMDT-Cirrhosis — Progression to Decompensation (survival redesign)

Rebuilt to fix the weak outcome and use the timing properly. Instead of predicting "first
cirrhosis ICD code" (a documentation event), this models **time to progression to
*decompensated* cirrhosis** — ascites, variceal bleeding, hepatic encephalopathy, SBP, or
hepatorenal syndrome — among **at-risk, non-cirrhotic** patients.

## What changed vs v1
- **Outcome:** time-to-**decompensation** (a concrete clinical event), not the cirrhosis code.
- **Richer observations:** added MELD physiology — Creatinine, INR, Sodium, Hemoglobin, WBC —
  on top of AST/ALT/Platelets/Albumin/Bilirubin (so the latent stage isn't just FIB-4).
- **Trajectory dynamics:** stage velocity, up-transitions, dwell time in the top stage, and
  stage-sequence entropy — "0→1→2" and "1→1→1" have the same peak but very different risk.
- **Survival model (not AUC):** a Cox proportional-hazards model with the latent stage as a
  covariate → **hazard ratios** and **concordance (C-index)**, benchmarked against FIB-4 and
  MELD-Na, plus **1-year decompensation risk by baseline stage**.
- The HMM stays **unsupervised** (glass-box); the outcome enters only through a **hazard layer
  on top**, so the staging is not contaminated by the label.

## Honest caveats
- MIMIC has short, sparse, irregular follow-up, so survival times are **heavily censored** and
  short — hazard estimates are noisier than a registry with real longitudinal follow-up.
- "Decompensation" is still ICD/marker-based, but far more concrete than a cirrhosis code.
- **Baseline-anchored** Cox (covariates from the first visit) is the clean model; the
  trajectory-augmented model peeks across pre-event visits, so it carries a mild
  immortal-time caveat — reported separately and flagged.

*Requires MIMIC-IV `hosp/`: diagnoses_icd, admissions, patients, labevents. Needs `lifelines`.*

## HMM library

In [2]:
"""Missing-aware diagonal-Gaussian HMM with optional treatment fusion.
Modes:
  baseline   : Gaussian emissions, homogeneous transitions
  emission   : Gaussian + Bernoulli(treatment) emissions, homogeneous transitions
  transition : Gaussian emissions, treatment-conditioned transitions (A_off / A_on)
"""
import numpy as np

VAR_FLOOR = 1e-3

def emission_logB(O, mu, var, u=None, p_treat=None):
    """O: T x D (NaN allowed). returns logB: T x K. Optional Bernoulli treatment emission."""
    T, D = O.shape; K = mu.shape[0]
    mask = ~np.isnan(O)                                  # T x D
    diff = O[:, None, :] - mu[None, :, :]                # T x K x D
    term = -0.5 * (np.log(2*np.pi*var)[None] + (diff**2)/var[None])
    term = np.where(mask[:, None, :], term, 0.0)
    logB = term.sum(axis=2)                              # T x K
    if u is not None and p_treat is not None:
        pt = np.clip(p_treat, 1e-4, 1-1e-4)              # K
        logB = logB + (u[:, None]*np.log(pt)[None] + (1-u[:, None])*np.log(1-pt)[None])
    return logB

def fwd_bwd(logB, pi, A_t):
    """A_t: (T-1) x K x K time-indexed transition matrices. Scaled forward-backward."""
    T, K = logB.shape
    m = logB.max(axis=1); B = np.exp(logB - m[:, None])
    alpha = np.zeros((T, K)); c = np.zeros(T)
    alpha[0] = pi * B[0]; c[0] = alpha[0].sum() + 1e-300; alpha[0] /= c[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A_t[t-1]) * B[t]
        c[t] = alpha[t].sum() + 1e-300; alpha[t] /= c[t]
    beta = np.zeros((T, K)); beta[-1] = 1.0
    for t in range(T-2, -1, -1):
        beta[t] = (A_t[t] @ (B[t+1]*beta[t+1])) / c[t+1]
    gamma = alpha*beta; gamma /= gamma.sum(axis=1, keepdims=True) + 1e-300
    loglik = np.sum(np.log(c)) + np.sum(m)
    return alpha, beta, gamma, B, c, loglik

def _init(seqs, K, seed=0):
    rng = np.random.default_rng(seed)
    allobs = np.vstack([s["O"] for s in seqs])
    D = allobs.shape[1]
    med = np.nanmedian(allobs, axis=0)
    # severity proxy = mean of standardized obs (already standardized upstream) -> quantile init
    sev = np.nanmean(allobs, axis=1)
    qs = np.nanquantile(sev, np.linspace(0.1, 0.9, K))
    mu = np.tile(med, (K, 1)).astype(float)
    # spread state means along each dim by quantiles of that dim
    for d in range(D):
        col = allobs[:, d]; col = col[~np.isnan(col)]
        if len(col):
            mu[:, d] = np.quantile(col, np.linspace(0.2, 0.8, K))
    var = np.tile(np.nanvar(allobs, axis=0), (K, 1)); var[var < VAR_FLOOR] = 1.0
    pi = np.full(K, 1.0/K)
    A = np.full((K, K), 0.1/(K-1)) + np.eye(K)*0.9
    A /= A.sum(axis=1, keepdims=True)
    return pi, A, mu, var

def fit(seqs, K=4, mode="baseline", n_iter=25, seed=0, verbose=False):
    """seqs: list of dict with 'O' (T x D) and optional 'u' (T,) treatment binary."""
    pi, A, mu, var = _init(seqs, K, seed)
    A_on = A.copy(); A_off = A.copy()
    p_treat = np.linspace(0.2, 0.8, K) if mode == "emission" else None
    D = mu.shape[1]
    prev_ll = -np.inf
    for it in range(n_iter):
        pi_acc = np.zeros(K)
        num_mu = np.zeros((K, D)); den_mu = np.zeros((K, D)); num_var = np.zeros((K, D))
        if mode == "transition":
            xi_off = np.zeros((K, K)); xi_on = np.zeros((K, K))
        else:
            xi_sum = np.zeros((K, K))
        tp_num = np.zeros(K); tp_den = np.zeros(K)
        total_ll = 0.0
        for s in seqs:
            O = s["O"]; T = O.shape[0]
            u = s.get("u") if mode != "baseline" else None
            logB = emission_logB(O, mu, var, u=(u if mode == "emission" else None),
                                 p_treat=(p_treat if mode == "emission" else None))
            if mode == "transition" and u is not None:
                A_t = np.stack([A_on if u[t] > 0.5 else A_off for t in range(T-1)]) if T > 1 \
                      else np.zeros((0, K, K))
            else:
                A_t = np.stack([A]*(T-1)) if T > 1 else np.zeros((0, K, K))
            alpha, beta, gamma, B, c, ll = fwd_bwd(logB, pi, A_t)
            total_ll += ll
            pi_acc += gamma[0]
            mask = ~np.isnan(O)
            # mean AND variance from the SAME posteriors (cached gamma)
            Osq = np.where(mask, O*O, 0.0)
            for k in range(K):
                g = gamma[:, k]
                num_mu[k]  += np.nansum(np.where(mask, O, 0.0) * g[:, None], axis=0)
                den_mu[k]  += (mask * g[:, None]).sum(axis=0)
                num_var[k] += (Osq * g[:, None]).sum(axis=0)   # sum gamma*x^2 (suff. stat)
            for t in range(T-1):
                At = (A_on if (mode == "transition" and u is not None and u[t] > 0.5)
                      else A_off if mode == "transition" else A)
                xi = (alpha[t][:, None] * At * (B[t+1]*beta[t+1])[None, :]) / (c[t+1])
                xi /= xi.sum() + 1e-300
                if mode == "transition" and u is not None:
                    if u[t] > 0.5: xi_on += xi
                    else: xi_off += xi
                else:
                    xi_sum += xi
            if mode == "emission" and u is not None:
                tp_num += (gamma * u[:, None]).sum(axis=0)
                tp_den += gamma.sum(axis=0)
        # M-step
        N = len(seqs)
        pi = pi_acc / N; pi /= pi.sum()
        mu = num_mu / np.maximum(den_mu, 1e-6)
        var = num_var / np.maximum(den_mu, 1e-6) - mu**2   # E[x^2]-E[x]^2
        var[var < VAR_FLOOR] = VAR_FLOOR
        if mode == "transition":
            A_off = xi_off / np.maximum(xi_off.sum(axis=1, keepdims=True), 1e-12)
            A_on  = xi_on  / np.maximum(xi_on.sum(axis=1, keepdims=True), 1e-12)
        else:
            A = xi_sum / np.maximum(xi_sum.sum(axis=1, keepdims=True), 1e-12)
        if mode == "emission":
            p_treat = np.clip(tp_num/np.maximum(tp_den, 1e-6), 1e-4, 1-1e-4)
        if verbose:
            print(f"  iter {it+1}: loglik={total_ll:.1f}")
        if abs(total_ll - prev_ll) < 1e-3*abs(prev_ll+1e-9):
            break
        prev_ll = total_ll
    params = dict(pi=pi, mu=mu, var=var, K=K, mode=mode)
    if mode == "transition": params["A_off"], params["A_on"] = A_off, A_on
    else: params["A"] = A
    if mode == "emission": params["p_treat"] = p_treat
    params["loglik"] = total_ll
    return params

def reorder_by_severity(params, direction):
    """direction: array length D, sign of severity per obs dim. Sort states ascending severity."""
    mu = params["mu"]; sev = (mu * direction[None, :]).sum(axis=1)
    order = np.argsort(sev)
    inv = np.argsort(order)
    p = dict(params)
    p["mu"] = mu[order]; p["var"] = params["var"][order]; p["pi"] = params["pi"][order]
    p["pi"] /= p["pi"].sum()
    if params["mode"] == "transition":
        p["A_off"] = params["A_off"][np.ix_(order, order)]
        p["A_on"]  = params["A_on"][np.ix_(order, order)]
    else:
        p["A"] = params["A"][np.ix_(order, order)]
    if "p_treat" in params: p["p_treat"] = params["p_treat"][order]
    p["_order"] = order
    return p

def posterior_stage(params, O, u=None):
    """Return gamma (T x K) and expected stage per window."""
    mode = params["mode"]; T = O.shape[0]
    logB = emission_logB(O, params["mu"], params["var"],
                         u=(u if mode == "emission" else None),
                         p_treat=(params.get("p_treat") if mode == "emission" else None))
    if mode == "transition" and u is not None:
        A_t = np.stack([params["A_on"] if u[t] > 0.5 else params["A_off"] for t in range(T-1)]) if T>1 else np.zeros((0,params["K"],params["K"]))
    else:
        A = params.get("A", params.get("A_off"))
        A_t = np.stack([A]*(T-1)) if T>1 else np.zeros((0,params["K"],params["K"]))
    _,_,gamma,_,_,ll = fwd_bwd(logB, params["pi"], A_t)
    stages = (gamma * np.arange(params["K"])[None, :]).sum(axis=1)
    return gamma, stages, ll

def obs_pred_loglik(params, seqs):
    """FAIR cross-mode metric: one-step-ahead predictive log-lik of the next
    observation vector, P(O_{t+1} | O_{1:t}, u_{1:t}). Gaussian (lab) part only,
    so emission/transition/baseline are scored on the SAME target."""
    mode = params["mode"]; K = params["K"]; mu = params["mu"]; var = params["var"]
    A_def = params.get("A", params.get("A_off"))
    tot = 0.0; nwin = 0
    for s in seqs:
        O = s["O"]; T = O.shape[0]; u = s.get("u")
        logB = emission_logB(O, mu, var,
                             u=(u if mode == "emission" else None),
                             p_treat=(params.get("p_treat") if mode == "emission" else None))
        m = logB.max(axis=1); B = np.exp(logB - m[:, None])      # full emission for filtering
        gaussB = np.exp(emission_logB(O, mu, var) -              # gaussian-only for prediction
                        emission_logB(O, mu, var).max(axis=1)[:, None])
        gm = emission_logB(O, mu, var).max(axis=1)
        alpha = params["pi"] * B[0]; alpha /= alpha.sum() + 1e-300
        for t in range(T-1):
            if mode == "transition" and u is not None:
                A = params["A_on"] if u[t] > 0.5 else params["A_off"]
            else:
                A = A_def
            pred = alpha @ A
            pdens = (pred * gaussB[t+1]).sum()                   # predictive density (scaled)
            tot += np.log(pdens + 1e-300) + gm[t+1]              # undo gauss scaling
            nwin += 1
            alpha = pred * B[t+1]; alpha /= alpha.sum() + 1e-300
    return tot, nwin, tot/max(nwin, 1)

def heldout_loglik(params, seqs):
    tot = 0.0; nwin = 0
    for s in seqs:
        u = s.get("u")
        _,_,ll = posterior_stage(params, s["O"], u=u)
        tot += ll; nwin += s["O"].shape[0]
    return tot, nwin, tot/max(nwin,1)


## Config

In [3]:
# ============================================================
# CONFIG
# ============================================================
import os, json, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

CANDIDATE_ROOTS = ["/Users/sumiyasayeed/Documents/mimic4/mimiciv",
                   "../data/mimiciv", "./data/mimiciv", "../../data/mimiciv"]
MIMIC_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.exists(os.path.join(p,"hosp"))),
                  CANDIDATE_ROOTS[0])
HOSP = os.path.join(MIMIC_ROOT, "hosp")
OUT = Path("results/progression_output"); OUT.mkdir(exist_ok=True)
CHUNK = 1_000_000
K_STAGE = 3
MIN_VISITS = 2
RANDOM_SEED = 1
print("MIMIC_ROOT =", MIMIC_ROOT, "| hosp exists:", os.path.exists(HOSP))

def _csv(d,n):
    for e in (".csv",".csv.gz"):
        if os.path.exists(os.path.join(d,n+e)): return os.path.join(d,n+e)
    return os.path.join(d,n+".csv")

RISK = {
  "HCV":            {9:("07044","07054","07070","07071"), 10:("B182","B1920","B1921")},
  "HBV":            {9:("07022","07023","07032","07033","07042","07052"), 10:("B180","B181","B1910","B1911")},
  "alcohol":        {9:("303","3050","5710","5711","5713"), 10:("F10","K700","K701","K709")},
  "NAFLD_NASH":     {9:("5718","5719"), 10:("K7581","K760","K7589")},
  "hemochromatosis":{9:("27501","27503"), 10:("E8310","E8311")},
}
CIRR = {9:("5712","5715","5716"),
        10:("K7030","K7031","K717","K740","K741","K742","K743","K744","K745","K746")}
# Decompensation = the concrete progression event (ascites/varices bleed/HE/SBP/HRS)
DECOMP = {9:("78959","7895","4560","45620","5722","56723","5724"),
          10:("R180","R188","R189","I8501","I8511","K720","K721","K729","K7290","K7291","K652","K767")}

LAB_IDS = {"AST":50878,"ALT":50861,"Platelets":51265,"Albumin":50862,"Bilirubin":50885,
           "Creatinine":50912,"INR":51237,"Sodium":50983,"Hemoglobin":51222,"WBC":51301}
OBS = list(LAB_IDS)                                   # 10-lab observation panel
LOG_OBS = {"AST","ALT","Bilirubin","Creatinine","WBC"}
SEV_DIR = {"AST":+1,"ALT":+1,"Platelets":-1,"Albumin":-1,"Bilirubin":+1,
           "Creatinine":+1,"INR":+1,"Sodium":-1,"Hemoglobin":-1,"WBC":+1}
print("Config loaded. obs =", OBS)


MIMIC_ROOT = /Users/sumiyasayeed/Documents/mimic4/mimiciv | hosp exists: True
Config loaded. obs = ['AST', 'ALT', 'Platelets', 'Albumin', 'Bilirubin', 'Creatinine', 'INR', 'Sodium', 'Hemoglobin', 'WBC']


## Step 1 — cohort + time-to-decompensation + labs

In [4]:
# ============================================================
# STEP 1 — at-risk cohort, time-to-DECOMPENSATION, per-visit labs
# ============================================================
def _pref(code_up, ver, p9, p10):
    return ((ver==9)&code_up.str.startswith(p9)) | ((ver==10)&code_up.str.startswith(p10))

dx = pd.read_csv(_csv(HOSP,"diagnoses_icd"), usecols=["subject_id","hadm_id","icd_code","icd_version"])
code = dx["icd_code"].astype(str).str.replace(".","",regex=False).str.upper()
dx["is_cirr"]   = _pref(code, dx["icd_version"], CIRR[9], CIRR[10]).astype(int)
dx["is_decomp"] = _pref(code, dx["icd_version"], DECOMP[9], DECOMP[10]).astype(int)
for n,c in RISK.items():
    dx[n] = _pref(code, dx["icd_version"], c[9], c[10]).astype(int)
perh = dx.groupby(["subject_id","hadm_id"])[["is_cirr","is_decomp"]+list(RISK)].max().reset_index()

adm = pd.read_csv(_csv(HOSP,"admissions"), usecols=["subject_id","hadm_id","admittime"])
adm["admittime"] = pd.to_datetime(adm["admittime"], errors="coerce")
adm = adm.merge(perh, on=["subject_id","hadm_id"], how="left").fillna(0).sort_values(["subject_id","admittime"])

base = adm.drop_duplicates("subject_id", keep="first").copy()
base["any_risk"] = base[list(RISK)].max(axis=1)
# at-risk AND not cirrhotic AND not already decompensated at baseline
cohort_subj = set(base[(base.any_risk==1) & (base.is_cirr==0) & (base.is_decomp==0)].subject_id)
adm_c = adm[adm.subject_id.isin(cohort_subj)].copy()
base_time = dict(zip(base.subject_id, base.admittime)); adm_c["base_time"] = adm_c.subject_id.map(base_time)

# event = first decompensation admission after baseline; else censor at last admission
post = adm_c[adm_c.admittime > adm_c.base_time]
first_dec = post[post.is_decomp==1].groupby("subject_id")["admittime"].min()
last_adm  = adm_c.groupby("subject_id")["admittime"].max()
surv = pd.DataFrame({"subject_id":list(cohort_subj)})
surv["base_time"] = surv.subject_id.map(base_time)
surv["dec_time"]  = surv.subject_id.map(first_dec)
surv["last_time"] = surv.subject_id.map(last_adm)
surv["event"] = surv["dec_time"].notna().astype(int)
end = surv["dec_time"].where(surv["event"]==1, surv["last_time"])
surv["duration_days"] = (end - surv["base_time"]).dt.total_seconds()/86400
surv = surv[surv["duration_days"] > 0]

# per-visit labs (pre-event)
hadm_set = set(adm_c.hadm_id.astype(int)); want = set(LAB_IDS.values())
acc = {}
for ch in pd.read_csv(_csv(HOSP,"labevents"), usecols=["hadm_id","itemid","valuenum"], chunksize=CHUNK, low_memory=False):
    ch = ch[ch.hadm_id.isin(hadm_set) & ch.itemid.isin(want) & ch.valuenum.notna()]
    if ch.empty: continue
    for (h,i),v in ch.groupby(["hadm_id","itemid"])["valuenum"].median().items():
        acc[(int(h),int(i))] = v
id2name = {v:k for k,v in LAB_IDS.items()}
labw = pd.DataFrame([{"hadm_id":h, **{id2name[i]:acc.get((h,i),np.nan) for i in want}} for h in hadm_set])
pts = pd.read_csv(_csv(HOSP,"patients"), usecols=["subject_id","anchor_age"])
visits = adm_c.merge(labw, on="hadm_id", how="left").merge(pts, on="subject_id", how="left")
visits = visits.merge(surv[["subject_id","dec_time","event","duration_days"]], on="subject_id", how="left")
visits = visits[visits.subject_id.isin(set(surv.subject_id))]
# censor visits at the decompensation event
keep = visits["dec_time"].isna() | (visits["admittime"] < visits["dec_time"])
visits = visits[keep].copy()
visits["FIB4"] = ((visits["anchor_age"]*visits["AST"]) /
                  (visits["Platelets"].clip(lower=1)*np.sqrt(visits["ALT"].clip(lower=1)))).clip(0,100)
# MELD-Na
b = visits["Bilirubin"].clip(lower=1); i_ = visits["INR"].clip(lower=1); c_ = visits["Creatinine"].clip(1,4)
meld = (3.78*np.log(b)+11.2*np.log(i_)+9.57*np.log(c_)+6.43).clip(6,40)
na = visits["Sodium"].clip(125,137)
visits["MELD_Na"] = (meld + 1.32*(137-na) - 0.033*meld*(137-na)).clip(6,40)

visits["has_obs"] = visits[OBS].notna().any(axis=1)
visits = visits[visits["has_obs"]].sort_values(["subject_id","admittime"])
vc = visits.groupby("subject_id").size(); serial = set(vc[vc>=MIN_VISITS].index)
visits = visits[visits.subject_id.isin(serial)]
surv = surv[surv.subject_id.isin(serial)]
print("="*60)
print(f"at-risk non-cirrhotic, not decompensated at baseline : {len(cohort_subj):,}")
print(f"  with >= {MIN_VISITS} pre-event lab visits ............ {visits.subject_id.nunique():,}")
print(f"  progress to DECOMPENSATION (events) ............ {int(surv.event.sum()):,} "
      f"({surv.event.mean()*100:.1f}%)")
print(f"  median follow-up (days) ........................ {surv.duration_days.median():.0f}")
print("="*60)


at-risk non-cirrhotic, not decompensated at baseline : 20,402
  with >= 2 pre-event lab visits ............ 5,597
  progress to DECOMPENSATION (events) ............ 301 (5.4%)
  median follow-up (days) ........................ 773


## Step 2 — HMM (richer obs) + trajectory dynamics

In [5]:
# ============================================================
# STEP 2 — unsupervised HMM (richer obs) + trajectory dynamics
# ============================================================
v = visits.copy()
for c in OBS:
    if c in LOG_OBS: v[c] = np.log1p(v[c].clip(lower=0))
STD = {}
for c in OBS:
    x = v[c].dropna().values
    if len(x) < 10: STD[c]=(0.0,1.0); continue
    med=float(np.median(x)); iqr=float(np.percentile(x,75)-np.percentile(x,25))
    sd=iqr/1.349 if iqr>1e-6 else float(np.std(x)); sd=sd if sd>1e-6 else 1.0
    STD[c]=(med,sd); v[c]=np.clip((v[c]-med)/sd,-8,8)

seqs=[]
for sid,g in v.groupby("subject_id"):
    O=g[OBS].values.astype(float)
    if len(O)<MIN_VISITS: continue
    seqs.append(dict(pid=sid,O=O,fib4=g["FIB4"].values,meld=g["MELD_Na"].values,
                     admit=pd.to_datetime(g["admittime"]).values))
t0=time.time()
params=fit(seqs,K=K_STAGE,mode="baseline",n_iter=30,seed=RANDOM_SEED)
sev=params["mu"] @ np.array([SEV_DIR[f] for f in OBS])
order=np.argsort(sev)
params=dict(params, mu=params["mu"][order], var=params["var"][order],
            pi=params["pi"][order]/params["pi"][order].sum(), A=params["A"][np.ix_(order,order)])
print(f"✅ HMM fit ({len(seqs):,} patients, {time.time()-t0:.1f}s); stage severity (asc):", np.round(sev[order],2))

def entropy(p):
    p=p[p>0]; return float(-(p*np.log(p)).sum()) if len(p) else 0.0
rows=[]
for s in seqs:
    g,st,_=posterior_stage(params,s["O"]); hard=np.argmax(g,axis=1)
    days=(pd.to_datetime(s["admit"]).astype("int64")/1e9/86400); span=max(np.ptp(days),1e-6)
    occ=np.bincount(hard, minlength=K_STAGE)/len(hard)
    rows.append(dict(pid=s["pid"],
        base_stage=float(st[0]), peak_stage=float(st.max()), last_stage=float(st[-1]), mean_stage=float(st.mean()),
        stage_velocity=float((st[-1]-st[0])/span*365),               # stages per year
        n_up=int((np.diff(hard)>0).sum()), dwell_top=float((hard==K_STAGE-1).mean()),
        stage_entropy=entropy(occ), n_visits=len(st),
        fib4_base=float(s["fib4"][0]) if np.isfinite(s["fib4"][0]) else np.nan,
        fib4_max=float(np.nanmax(s["fib4"])) if np.isfinite(s["fib4"]).any() else np.nan,
        meld_base=float(s["meld"][0]) if np.isfinite(s["meld"][0]) else np.nan,
        meld_max=float(np.nanmax(s["meld"])) if np.isfinite(s["meld"]).any() else np.nan))
SF=pd.DataFrame(rows).merge(surv[["subject_id","event","duration_days"]].rename(columns={"subject_id":"pid"}), on="pid")
print("\nDecompensation rate by baseline latent stage:")
print(SF.groupby(SF.base_stage.round()).agg(n=("pid","size"), event_rate=("event","mean"),
        median_FU=("duration_days","median")).round(3).to_string())


✅ HMM fit (5,597 patients, 2.8s); stage severity (asc): [-1.12  2.49  7.63]

Decompensation rate by baseline latent stage:
               n  event_rate  median_FU
base_stage                             
0.0         3352       0.033    919.608
1.0         1305       0.076    586.776
2.0          940       0.099    576.868


In [6]:
# ============================================================
# STEP 2b — grouped held-out patient split (parity with acute tasks)
# Fit the latent stage on TRAIN patients only; evaluate on TEST patients.
# ============================================================
rng = np.random.default_rng(RANDOM_SEED)
_pids = np.array([s["pid"] for s in seqs])
_perm = rng.permutation(len(_pids))
_n_test = int(0.30 * len(_pids))
_test_ids = set(_pids[_perm[:_n_test]])
train_seqs = [s for s in seqs if s["pid"] not in _test_ids]
test_seqs  = [s for s in seqs if s["pid"] in _test_ids]

# refit the HMM on train only, then re-order stages by severity (mirrors cell 8)
params_tr = fit(train_seqs, K=K_STAGE, mode="baseline", n_iter=30, seed=RANDOM_SEED)
_sev = params_tr["mu"] @ np.array([SEV_DIR[f] for f in OBS])
_o = np.argsort(_sev)
params_tr = dict(params_tr, mu=params_tr["mu"][_o], var=params_tr["var"][_o],
                 pi=params_tr["pi"][_o]/params_tr["pi"][_o].sum(),
                 A=params_tr["A"][np.ix_(_o, _o)])

def stage_features(P, S):
    rows = []
    for s in S:
        g, st, _ = posterior_stage(P, s["O"]); hard = np.argmax(g, axis=1)
        days = (pd.to_datetime(s["admit"]).astype("int64")/1e9/86400)
        span = max(np.ptp(days), 1e-6); occ = np.bincount(hard, minlength=K_STAGE)/len(hard)
        rows.append(dict(pid=s["pid"], base_stage=float(st[0]), peak_stage=float(st.max()),
            last_stage=float(st[-1]), mean_stage=float(st.mean()),
            stage_velocity=float((st[-1]-st[0])/span*365),
            n_up=int((np.diff(hard) > 0).sum()), dwell_top=float((hard == K_STAGE-1).mean()),
            stage_entropy=entropy(occ), n_visits=len(st),
            fib4_base=float(s["fib4"][0]) if np.isfinite(s["fib4"][0]) else np.nan,
            fib4_max=float(np.nanmax(s["fib4"])) if np.isfinite(s["fib4"]).any() else np.nan,
            meld_base=float(s["meld"][0]) if np.isfinite(s["meld"][0]) else np.nan,
            meld_max=float(np.nanmax(s["meld"])) if np.isfinite(s["meld"]).any() else np.nan))
    return pd.DataFrame(rows).merge(
        surv[["subject_id", "event", "duration_days"]].rename(columns={"subject_id": "pid"}), on="pid")

SF_train = stage_features(params_tr, train_seqs)
SF_test  = stage_features(params_tr, test_seqs)
print(f"grouped split: train={len(SF_train):,}  test={len(SF_test):,} patients "
      f"(HMM fit on train only)")
# To report HELD-OUT numbers, re-run STEP 3 (cell 11) with:  SF = SF_test

grouped split: train=3,918  test=1,679 patients (HMM fit on train only)


In [7]:
%pip install lifelines -q

Note: you may need to restart the kernel to use updated packages.


## Step 3 — survival model (Cox + 1-yr risk by stage)

**Model of record.** The single-split survival output below (n≈1,057; 23 one-year events) is underpowered and non-significant. All manuscript numbers come from the grouped five-fold out-of-fold (OOF) block further down (STEP 3c: n≈3,611; 79 events): stage risks 0.6 / 3.2 / 4.9%; 1-yr AUC 0.79 vs 0.68 (MELD-Na) / 0.62 (FIB-4); ΔAUC +0.11 [+0.05, +0.18] and +0.17 [+0.10, +0.24]; C-index 0.74 vs 0.65 / 0.67; Brier 0.021.

In [8]:
# ============================================================
# STEP 3 — SURVIVAL: 1-yr risk by stage + 1-yr discrimination
# (dependency-free; uses lifelines for full Cox HR/C-index if available)
# ============================================================
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_predict
try:
    from lifelines import CoxPHFitter
    from lifelines.utils import concordance_index
    HAVE_LL = True
except Exception:
    HAVE_LL = False; print("lifelines not installed -> using dependency-free survival fallback\n")

D = SF_test.copy(); D["dur"] = D["duration_days"].clip(lower=1)

# ---- manual Kaplan-Meier 1-year risk by baseline stage ----
def km_risk(dur, ev, t=365):
    dur = np.asarray(dur, float); ev = np.asarray(ev, int)
    surv = 1.0
    for ut in np.unique(dur[ev == 1]):
        if ut > t: break
        d = int(((dur == ut) & (ev == 1)).sum()); r = int((dur >= ut).sum())
        if r > 0: surv *= (1 - d/r)
    return 1 - surv
print("=== 1-year decompensation risk by baseline latent stage (Kaplan-Meier) ===")
for k in range(K_STAGE):
    m = D["base_stage"].round() == k
    if m.sum() < 20: continue
    print(f"  baseline stage {k}: n={int(m.sum()):<5} 1-year risk = {km_risk(D.loc[m,'dur'], D.loc[m,'event'])*100:.1f}%")

# ---- 1-year discrimination benchmark (cases<=365d vs event-free>=365d) ----
lab = np.where((D.event==1)&(D.dur<=365), 1, np.where(D.dur>=365, 0, -1))
sel = lab >= 0; y = lab[sel]; Dd = D[sel].copy()
print(f"\n=== 1-year decompensation discrimination (n={int(sel.sum())}, events={int(y.sum())}) ===")
_DISC = (len(np.unique(y)) > 1 and y.sum() >= 10)
if not _DISC:
    print("  too few within-1-year events to benchmark discrimination (skipped)")
def boot(yt,s,n=1000,seed=42):
    rng=np.random.default_rng(seed); a=[]; yt=np.asarray(yt); s=np.asarray(s)
    for _ in range(n):
        idx=rng.integers(0,len(yt),len(yt))
        if not (1<yt[idx].sum()<len(idx)-1): continue
        a.append(roc_auc_score(yt[idx],s[idx]))
    return (np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)) if a else (np.nan,)*3
for name,col in [("MELD-Na max","meld_max"),("FIB-4 max","fib4_max"),
                 ("Peak stage","peak_stage"),("Stage velocity","stage_velocity"),("Dwell top","dwell_top")]:
    if not _DISC or col not in Dd.columns: continue
    m=Dd[col].notna().values
    if m.sum()<20 or len(np.unique(y[m]))<2: continue
    md,lo,hi=boot(y[m],Dd[col].values[m]); print(f"  {name:<16} AUC={md:.3f} [{lo:.3f}-{hi:.3f}]")
def cv_auc(cols):
    cc=[c for c in cols if c in Dd.columns]
    X=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(Dd[cc]))
    p=cross_val_predict(LogisticRegression(max_iter=2000,class_weight='balanced'),X,y,cv=5,method='predict_proba')[:,1]
    return roc_auc_score(y,p)
sfeat=["peak_stage","stage_velocity","dwell_top","stage_entropy"]
if _DISC:
    print(f"  {'Stage (combined)':<16} AUC={cv_auc(sfeat):.3f}")
    print(f"  {'Stage+MELD+FIB-4':<16} AUC={cv_auc(sfeat+['meld_max','fib4_max']):.3f}")

    # ---- paired bootstrap: AUC difference on the SAME patients ----
    from sklearn.model_selection import cross_val_predict as _cvp
    def _cv_scores(cols):
        cc=[c for c in cols if c in Dd.columns]
        Xs=StandardScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(Dd[cc]))
        return _cvp(LogisticRegression(max_iter=2000,class_weight='balanced'),Xs,y,cv=5,method='predict_proba')[:,1]
    p_stage = _cv_scores(sfeat)
    p_meld  = Dd["meld_max"].values.astype(float)
    p_fib   = Dd["fib4_max"].values.astype(float)
    def _paired_delta(yt, s_better, s_ref, n=2000, seed=42):
        m = ~(np.isnan(s_better)|np.isnan(s_ref)); yt2=np.asarray(yt)[m]; a=np.asarray(s_better)[m]; b=np.asarray(s_ref)[m]
        rng=np.random.default_rng(seed); d=[]
        for _ in range(n):
            idx=rng.integers(0,len(yt2),len(yt2))
            if not (1<yt2[idx].sum()<len(idx)-1): continue
            d.append(roc_auc_score(yt2[idx],a[idx])-roc_auc_score(yt2[idx],b[idx]))
        return (np.median(d),np.percentile(d,2.5),np.percentile(d,97.5)) if d else (np.nan,)*3
    print("\n=== Paired bootstrap: Stage(combined) - benchmark (same patients) ===")
    for ref_name, ref in [("MELD-Na max", p_meld), ("FIB-4 max", p_fib)]:
        md,lo,hi=_paired_delta(y, p_stage, ref)
        sig="** significant" if (lo>0 or hi<0) else "ns (CI crosses 0)"
        print(f"  Stage - {ref_name:<12} = {md:+.3f} [95% CI {lo:+.3f}, {hi:+.3f}]  {sig}")

# ---- full Cox (hazard ratios + C-index) if lifelines available ----
if HAVE_LL:
    def cox(cols,label):
        cc=[c for c in cols if c in D.columns]; d=D[cc+["dur","event"]].dropna()
        if d.event.sum()<10 or len(d)<50: print(f"  [{label}] too few"); return
        for c in cc: d[c]=(d[c]-d[c].mean())/(d[c].std()+1e-9)
        cph=CoxPHFitter(penalizer=0.1).fit(d,"dur","event")
        ci=concordance_index(d["dur"],-cph.predict_partial_hazard(d),d["event"])
        print(f"\n[{label}] C-index={ci:.3f} (n={len(d)}, events={int(d.event.sum())})")
        h=cph.summary[["exp(coef)","exp(coef) lower 95%","exp(coef) upper 95%","p"]]; h.columns=["HR","lo","hi","p"]
        print(h.round(3).to_string())
    print("\n=== Cox proportional hazards (HR per 1 SD) ===")
    cox(["meld_max"],"MELD-Na"); cox(["fib4_max"],"FIB-4")
    cox(["peak_stage","stage_velocity","dwell_top","stage_entropy"],"Latent stage + dynamics")
    cox(["base_stage","fib4_base","meld_base"],"Baseline-only (clean)")
SF.to_csv(OUT/"progression_survival_features.csv", index=False)
print("\n✅ Saved:", OUT/"progression_survival_features.csv")


=== 1-year decompensation risk by baseline latent stage (Kaplan-Meier) ===
  baseline stage 0: n=1034  1-year risk = 0.7%
  baseline stage 1: n=352   1-year risk = 4.0%
  baseline stage 2: n=293   1-year risk = 3.4%

=== 1-year decompensation discrimination (n=1057, events=23) ===
  MELD-Na max      AUC=0.609 [0.471-0.739]
  FIB-4 max        AUC=0.627 [0.489-0.744]
  Peak stage       AUC=0.683 [0.572-0.774]
  Stage velocity   AUC=0.592 [0.422-0.742]
  Dwell top        AUC=0.654 [0.528-0.779]
  Stage (combined) AUC=0.662
  Stage+MELD+FIB-4 AUC=0.624

=== Paired bootstrap: Stage(combined) - benchmark (same patients) ===
  Stage - MELD-Na max  = +0.032 [95% CI -0.153, +0.193]  ns (CI crosses 0)
  Stage - FIB-4 max    = -0.005 [95% CI -0.197, +0.174]  ns (CI crosses 0)

=== Cox proportional hazards (HR per 1 SD) ===

[MELD-Na] C-index=0.642 (n=1228, events=77)
              HR     lo     hi      p
covariate                            
meld_max   1.179  1.034  1.343  0.014

[FIB-4] C-index=

In [9]:
# ============================================================
# STEP 3c — GROUPED K-FOLD OUT-OF-FOLD (leakage-free AND fully powered)
# Refit the HMM on train folds, score the held-out fold, pool all patients.
# Uses every event for evaluation while keeping train/test separation:
# fair like SF_test, powered like the full cohort. Needs `stage_features`
# (cell 9) and the imports/HAVE_LL from cell 12.
# ============================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, brier_score_loss

K_FOLDS = 5
_rng = np.random.default_rng(RANDOM_SEED)
_idx = _rng.permutation(len(seqs))
_folds = np.array_split(_idx, K_FOLDS)

_oof = []
for f in range(K_FOLDS):
    _te = set(_folds[f].tolist())
    _tr_seqs = [seqs[i] for i in range(len(seqs)) if i not in _te]
    _te_seqs = [seqs[i] for i in range(len(seqs)) if i in _te]
    _P = fit(_tr_seqs, K=K_STAGE, mode="baseline", n_iter=30, seed=RANDOM_SEED)
    _sev = _P["mu"] @ np.array([SEV_DIR[c] for c in OBS]); _o = np.argsort(_sev)
    _P = dict(_P, mu=_P["mu"][_o], var=_P["var"][_o],
              pi=_P["pi"][_o] / _P["pi"][_o].sum(), A=_P["A"][np.ix_(_o, _o)])
    _oof.append(stage_features(_P, _te_seqs))
SF_oof = pd.concat(_oof, ignore_index=True)
Do = SF_oof.copy(); Do["dur"] = Do["duration_days"].clip(lower=1)
print(f"OOF pooled: {len(Do):,} patients from {K_FOLDS} grouped folds")

def _km(dur, ev, t=365):
    dur = np.asarray(dur, float); ev = np.asarray(ev, int); s = 1.0
    for ut in np.unique(dur[ev == 1]):
        if ut > t: break
        d = int(((dur == ut) & (ev == 1)).sum()); r = int((dur >= ut).sum())
        if r > 0: s *= (1 - d / r)
    return 1 - s
print("\n1-year decompensation risk by baseline stage (OOF):")
for k in range(K_STAGE):
    m = Do["base_stage"].round() == k
    if m.sum() < 20: continue
    print(f"  stage {k}: n={int(m.sum()):<5} 1-yr risk = {_km(Do.loc[m,'dur'], Do.loc[m,'event'])*100:.1f}%")

_lab = np.where((Do.event == 1) & (Do.dur <= 365), 1, np.where(Do.dur >= 365, 0, -1))
_sel = _lab >= 0; yo = _lab[_sel]; Dq = Do[_sel].copy()
print(f"\n1-year discrimination (OOF): n={int(_sel.sum())}, events={int(yo.sum())}")

def _boot(yt, s, n=2000, seed=42):
    rng = np.random.default_rng(seed); a = []; yt = np.asarray(yt); s = np.asarray(s)
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        if 1 < yt[i].sum() < len(i) - 1: a.append(roc_auc_score(yt[i], s[i]))
    return (np.median(a), np.percentile(a, 2.5), np.percentile(a, 97.5)) if a else (np.nan,) * 3
for nm, col in [("MELD-Na max", "meld_max"), ("FIB-4 max", "fib4_max"),
                ("Peak stage", "peak_stage"), ("Dwell top", "dwell_top")]:
    if col not in Dq.columns: continue
    m = Dq[col].notna().values
    if m.sum() < 20 or len(np.unique(yo[m])) < 2: continue
    md, lo, hi = _boot(yo[m], Dq[col].values[m]); print(f"  {nm:<14} AUC={md:.3f} [{lo:.3f}-{hi:.3f}]")

_sfeat = ["peak_stage", "stage_velocity", "dwell_top", "stage_entropy"]
_Xs = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Dq[_sfeat]))
p_stage = cross_val_predict(LogisticRegression(max_iter=2000, class_weight="balanced"),
                            _Xs, yo, cv=5, method="predict_proba")[:, 1]
print(f"  {'Stage (combined)':<14} AUC={roc_auc_score(yo, p_stage):.3f}")

def _pd(yt, a, b, n=2000, seed=42):
    m = ~(np.isnan(a) | np.isnan(b)); yt = np.asarray(yt)[m]; a = np.asarray(a)[m]; b = np.asarray(b)[m]
    rng = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = rng.integers(0, len(yt), len(yt))
        if 1 < yt[i].sum() < len(i) - 1: d.append(roc_auc_score(yt[i], a[i]) - roc_auc_score(yt[i], b[i]))
    return (np.median(d), np.percentile(d, 2.5), np.percentile(d, 97.5)) if d else (np.nan,) * 3
print("\nPaired bootstrap Delta-AUC (OOF, same patients):")
for nm, ref in [("MELD-Na", Dq["meld_max"].values.astype(float)),
                ("FIB-4", Dq["fib4_max"].values.astype(float))]:
    md, lo, hi = _pd(yo, p_stage, ref)
    sig = "** significant" if (lo > 0 or hi < 0) else "ns (CI crosses 0)"
    print(f"  Stage(combined) - {nm:<8} = {md:+.3f} [95% CI {lo:+.3f}, {hi:+.3f}]  {sig}")

if HAVE_LL:
    def _ci(cols, label):
        cc = [c for c in cols if c in Do.columns]; d = Do[cc + ["dur", "event"]].dropna()
        for c in cc: d[c] = (d[c] - d[c].mean()) / (d[c].std() + 1e-9)
        cph = CoxPHFitter(penalizer=0.1).fit(d, "dur", "event")
        ci = concordance_index(d["dur"], -cph.predict_partial_hazard(d), d["event"])
        print(f"  [{label}] C-index={ci:.3f} (events={int(d.event.sum())})")
    print("\nCox C-index (OOF):")
    _ci(["meld_max"], "MELD-Na"); _ci(["fib4_max"], "FIB-4"); _ci(_sfeat, "Latent stage + dynamics")

p_cal = cross_val_predict(LogisticRegression(max_iter=2000), _Xs, yo, cv=5, method="predict_proba")[:, 1]
print(f"\n1-year calibration (OOF, Brier={brier_score_loss(yo, p_cal):.3f})")
SF_oof.to_csv(OUT / "progression_oof_features.csv", index=False)
print("Saved:", OUT / "progression_oof_features.csv")


OOF pooled: 5,597 patients from 5 grouped folds

1-year decompensation risk by baseline stage (OOF):
  stage 0: n=3365  1-yr risk = 0.6%
  stage 1: n=1280  1-yr risk = 3.2%
  stage 2: n=952   1-yr risk = 4.9%

1-year discrimination (OOF): n=3611, events=79
  MELD-Na max    AUC=0.675 [0.619-0.731]
  FIB-4 max      AUC=0.616 [0.544-0.685]
  Peak stage     AUC=0.737 [0.677-0.788]
  Dwell top      AUC=0.718 [0.653-0.780]
  Stage (combined) AUC=0.790

Paired bootstrap Delta-AUC (OOF, same patients):
  Stage(combined) - MELD-Na  = +0.114 [95% CI +0.049, +0.180]  ** significant
  Stage(combined) - FIB-4    = +0.171 [95% CI +0.097, +0.242]  ** significant

Cox C-index (OOF):
  [MELD-Na] C-index=0.654 (events=269)
  [FIB-4] C-index=0.665 (events=280)
  [Latent stage + dynamics] C-index=0.737 (events=301)

1-year calibration (OOF, Brier=0.021)
Saved: results/progression_output/progression_oof_features.csv


## STEP 3d — OPTION A: baseline-only prognostic model (no look-ahead)

Uses only information available at the **first modeled admission**: the *filtered* latent-stage posterior
`alpha[0]` (first-admission labs + fold-trained parameters), baseline MELD-Na, and baseline FIB-4.
The one-year clock starts at that admission. Trajectory features (peak/velocity/dwell/entropy) are **not** used.
Requires cells 2, 8, 9 (for `entropy`, `surv`, `seqs`) and 12/14 (`HAVE_LL`, `_folds`, imports) to have been run.

In [10]:
# ============================================================
# STEP 3d — OPTION A: BASELINE-ONLY, FILTERED STAGE AT FIRST ADMISSION
# ============================================================
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from pathlib import Path
if "OUT" not in globals(): OUT = Path("results/progression_output"); OUT.mkdir(parents=True, exist_ok=True)
if "FIG" not in globals(): FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)

def baseline_filtered_stage(P, O):
    """Filtered posterior at t=0: alpha[0] ∝ pi * B[0]; uses ONLY the first admission's labs."""
    logB0 = emission_logB(O[:1], P["mu"], P["var"])[0]
    a0 = P["pi"] * np.exp(logB0 - logB0.max()); a0 = a0 / (a0.sum() + 1e-300)
    return a0

_rows = []
for f in range(K_FOLDS):
    _te = set(_folds[f].tolist())
    _tr_seqs = [seqs[i] for i in range(len(seqs)) if i not in _te]
    _te_seqs = [seqs[i] for i in range(len(seqs)) if i in _te]
    _P = fit(_tr_seqs, K=K_STAGE, mode="baseline", n_iter=30, seed=RANDOM_SEED)
    _sev = _P["mu"] @ np.array([SEV_DIR[c] for c in OBS]); _o = np.argsort(_sev)
    _P = dict(_P, mu=_P["mu"][_o], var=_P["var"][_o],
              pi=_P["pi"][_o] / _P["pi"][_o].sum(), A=_P["A"][np.ix_(_o, _o)])
    for s in _te_seqs:
        a0 = baseline_filtered_stage(_P, s["O"])
        _rows.append(dict(pid=s["pid"], fold=f,
            base_stage_filt=float((a0 * np.arange(K_STAGE)).sum()),
            base_stage_hard=int(np.argmax(a0)),
            base_p_top=float(a0[K_STAGE-1]),
            fib4_base=float(s["fib4"][0]) if np.isfinite(s["fib4"][0]) else np.nan,
            meld_base=float(s["meld"][0]) if np.isfinite(s["meld"][0]) else np.nan))
BL = pd.DataFrame(_rows).merge(surv[["subject_id","event","duration_days"]]
                               .rename(columns={"subject_id":"pid"}), on="pid")
BL["dur"] = BL["duration_days"].clip(lower=1)
print(f"Baseline-only OOF table: {len(BL):,} patients, {int(BL.event.sum())} events")
print("Missing baseline scores: MELD-Na", int(BL.meld_base.isna().sum()), "| FIB-4", int(BL.fib4_base.isna().sum()))

# ---- 1-year KM risk by baseline (filtered, hard) stage, all patients ----
print("\n1-year decompensation risk by baseline latent stage (filtered posterior at first admission):")
for k in range(K_STAGE):
    m = BL["base_stage_hard"] == k
    if m.sum() < 20: continue
    print(f"  stage {k}: n={int(m.sum()):<5} 1-yr risk = {_km(BL.loc[m,'dur'], BL.loc[m,'event'])*100:.1f}%")

# ---- 1-year label ----
_lab = np.where((BL.event == 1) & (BL.dur <= 365), 1, np.where(BL.dur >= 365, 0, -1))
E = BL[_lab >= 0].copy(); E["y1"] = _lab[_lab >= 0]
print(f"\n1-year eligible: n={len(E)}, events={int(E.y1.sum())}")

# AUC of each baseline score on ALL patients where available
print("\n1-year AUC (OOF), each score on all patients with that score available:")
for nm, col in [("Baseline latent stage", "base_stage_filt"), ("MELD-Na (baseline)", "meld_base"), ("FIB-4 (baseline)", "fib4_base")]:
    m = E[col].notna().values
    md, lo, hi = _boot(E.y1.values[m], E[col].values[m])
    print(f"  {nm:<24} AUC={md:.3f} [{lo:.3f}-{hi:.3f}]  (n={int(m.sum())}, events={int(E.y1.values[m].sum())})")

# Paired comparison on the COMMON subset (all three baseline scores present)
Cm = E.dropna(subset=["base_stage_filt","meld_base","fib4_base"]).copy()
yc = Cm.y1.values
print(f"\nPaired comparison on common subset: n={len(Cm)}, events={int(yc.sum())}")
for nm, col in [("Baseline latent stage", "base_stage_filt"), ("MELD-Na (baseline)", "meld_base"), ("FIB-4 (baseline)", "fib4_base")]:
    md, lo, hi = _boot(yc, Cm[col].values); print(f"  {nm:<24} AUC={md:.3f} [{lo:.3f}-{hi:.3f}]")
for nm, ref in [("MELD-Na", "meld_base"), ("FIB-4", "fib4_base")]:
    md, lo, hi = _pd(yc, Cm["base_stage_filt"].values, Cm[ref].values.astype(float))
    sig = "** significant" if (lo > 0 or hi < 0) else "ns (CI crosses 0)"
    print(f"  Stage - {nm:<8} = {md:+.3f} [95% CI {lo:+.3f}, {hi:+.3f}]  {sig}")

# Baseline combined logistic (stage + MELD-Na + FIB-4), OOF on common subset
_Xb = StandardScaler().fit_transform(Cm[["base_stage_filt","meld_base","fib4_base"]].values)
_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
p_comb = cross_val_predict(LogisticRegression(max_iter=2000), _Xb, yc, cv=_cv, method="predict_proba")[:, 1]
md, lo, hi = _boot(yc, p_comb)
print(f"  Stage + MELD-Na + FIB-4  AUC={md:.3f} [{lo:.3f}-{hi:.3f}]")
md, lo, hi = _pd(yc, p_comb, Cm["meld_base"].values.astype(float))
print(f"  (Stage+MELD-Na+FIB-4) - MELD-Na = {md:+.3f} [95% CI {lo:+.3f}, {hi:+.3f}]")

# Calibration of the baseline stage alone (OOF logistic on all eligible patients)
_Xs1 = E[["base_stage_filt"]].values
p_cal1 = cross_val_predict(LogisticRegression(max_iter=2000), _Xs1, E.y1.values, cv=_cv, method="predict_proba")[:, 1]
print(f"\n1-year calibration, baseline stage alone (OOF): Brier={brier_score_loss(E.y1.values, p_cal1):.3f}  (event rate {E.y1.mean():.3f})")

# ---- Cox, baseline only ----
if HAVE_LL:
    def _cox(cols, label, data):
        d = data[cols + ["dur", "event"]].dropna().copy()
        for c in cols: d[c] = (d[c] - d[c].mean()) / (d[c].std() + 1e-9)
        cph = CoxPHFitter(penalizer=0.1).fit(d, "dur", "event")
        ci = concordance_index(d["dur"], -cph.predict_partial_hazard(d), d["event"])
        print(f"\n  [{label}] n={len(d)} events={int(d.event.sum())} C-index={ci:.3f}")
        print(cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(3).to_string())
    print("\nCox proportional hazards, baseline covariates only (per SD):")
    _cox(["base_stage_filt"], "Baseline latent stage", BL)
    _cox(["meld_base"], "MELD-Na (baseline)", BL)
    _cox(["fib4_base"], "FIB-4 (baseline)", BL)
    _cox(["base_stage_filt", "meld_base", "fib4_base"], "Stage + MELD-Na + FIB-4", BL)

# ---- Figures (baseline versions) ----
plt.figure(figsize=(5.5, 4))
_t = np.arange(0, 366)
for k in range(K_STAGE):
    m = BL["base_stage_hard"] == k
    if m.sum() < 20: continue
    _risk = [_km(BL.loc[m, "dur"], BL.loc[m, "event"], t=tt) for tt in _t]
    plt.step(_t, np.array(_risk) * 100, where="post", label=f"baseline stage {k} (n={int(m.sum())})")
plt.xlabel("Days from first modeled admission"); plt.ylabel("Cumulative decompensation (%)")
plt.legend(frameon=False); plt.tight_layout(); plt.savefig(FIG / "km_by_stage_baseline.png", dpi=300); plt.show()

plt.figure(figsize=(4.5, 4.5))
for nm, col in [("Baseline latent stage", "base_stage_filt"), ("MELD-Na (baseline)", "meld_base"), ("FIB-4 (baseline)", "fib4_base")]:
    fpr, tpr, _ = roc_curve(yc, Cm[col].values); plt.plot(fpr, tpr, label=f"{nm} (AUC {roc_auc_score(yc, Cm[col].values):.2f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8); plt.xlabel("1 - specificity"); plt.ylabel("Sensitivity")
plt.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.savefig(FIG / "roc_stage_vs_scores_baseline.png", dpi=300); plt.show()

# ---- Calibration figure (baseline stage alone, OOF, quintile bins) ----
from sklearn.calibration import calibration_curve
_pt, _pp = calibration_curve(E.y1.values, p_cal1, n_bins=5, strategy="quantile")
plt.figure(figsize=(4.5, 4.5))
plt.plot([0, max(_pp.max(), _pt.max())*1.1], [0, max(_pp.max(), _pt.max())*1.1], "k--", lw=0.8, label="perfect")
plt.plot(_pp, _pt, "o-", label=f"baseline stage (Brier {brier_score_loss(E.y1.values, p_cal1):.3f})")
plt.xlabel("Predicted 1-year risk"); plt.ylabel("Observed 1-year risk"); plt.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "calibration_1yr_baseline.png", dpi=300); plt.show()
print("Saved figure calibration_1yr_baseline.png")

BL.to_csv(OUT / "progression_oof_baseline.csv", index=False)
print("\nSaved:", OUT / "progression_oof_baseline.csv", "and figures km_by_stage_baseline.png, roc_stage_vs_scores_baseline.png")


Baseline-only OOF table: 5,597 patients, 301 events
Missing baseline scores: MELD-Na 2998 | FIB-4 2248

1-year decompensation risk by baseline latent stage (filtered posterior at first admission):
  stage 0: n=3830  1-yr risk = 0.9%
  stage 1: n=690   1-yr risk = 4.4%
  stage 2: n=1077  1-yr risk = 4.1%

1-year eligible: n=3611, events=79

1-year AUC (OOF), each score on all patients with that score available:
  Baseline latent stage    AUC=0.713 [0.651-0.771]  (n=3611, events=79)
  MELD-Na (baseline)       AUC=0.664 [0.591-0.736]  (n=1530, events=52)
  FIB-4 (baseline)         AUC=0.640 [0.565-0.714]  (n=2048, events=60)

Paired comparison on common subset: n=1514, events=52
  Baseline latent stage    AUC=0.687 [0.613-0.762]
  MELD-Na (baseline)       AUC=0.665 [0.594-0.733]
  FIB-4 (baseline)         AUC=0.596 [0.511-0.676]
  Stage - MELD-Na  = +0.022 [95% CI -0.034, +0.075]  ns (CI crosses 0)
  Stage - FIB-4    = +0.092 [95% CI +0.006, +0.182]  ** significant
  Stage + MELD-Na + FIB

## STEP R — REVISION ANALYSES (IBMED-D-26-00792)

Run **after** cells up to and including STEP 3d (they create `visits`, `surv`, `seqs`, `_folds`,
`BL`, `E`, `Cm`, `p_cal1`, `baseline_filtered_stage`, `_km`, `_boot`, `_pd`, `HAVE_LL`).
Outputs: `results/progression_output/revision/` (git-ignored).

| Cell | Reviewer point |
|---|---|
| Ra cohort flow, included vs excluded, missing labs | R1.6, R2.4 |
| Rb fold-internal standardisation (removes the last pre-processing leakage) + K = 2..5 | R1.8, R1.9 |
| Rc censoring-aware discrimination (IPCW AUC, C-index with CIs), missing-lab sensitivity | R2.4, R1 minor |
| Rd operating characteristics at pre-specified thresholds | R1.5, R2.2 |
| Re supervised baselines (LR, RF, gradient boosting) and static-mixture ablation | R1.2, R1.3 |
| Rf stability of learned stages, attribution fidelity | R1.7, R1.8 |
| Rg temporal validation (anchor_year_group) | R1.10 |

Expected run time: 30 to 90 minutes (dominated by the 20 + 5 + 3 + 1 EM fits in Rb, Re, Rf, Rg).

In [11]:
# ============================================================
# STEP Ra — cohort flow, included vs excluded, missing labs
# ============================================================
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import revision_utils as ru
from sklearn.metrics import cohen_kappa_score
REVC = OUT / "revision"; REVC.mkdir(parents=True, exist_ok=True)
SEVV = np.array([SEV_DIR[c] for c in OBS])
K_GRID = [2, 3, 4, 5]

_pat = pd.read_csv(_csv(HOSP, "patients"))
_pat = _pat[[c for c in ["subject_id", "gender", "anchor_age", "anchor_year_group"] if c in _pat.columns]]

# cohort flow (recomputed from the STEP 1 objects, before the >= 2-visit filter)
_b = base.copy(); _b["any_risk"] = _b[list(RISK)].max(axis=1)
n_first = len(_b)
n_risk = int((_b.any_risk == 1).sum())
n_risk_cirr = int(((_b.any_risk == 1) & (_b.is_cirr == 1)).sum())
n_risk_dec = int(((_b.any_risk == 1) & (_b.is_cirr == 0) & (_b.is_decomp == 1)).sum())
_post = adm_c[adm_c.admittime > adm_c.base_time]
_first_dec = _post[_post.is_decomp == 1].groupby("subject_id")["admittime"].min()
_last = adm_c.groupby("subject_id")["admittime"].max()
S0 = pd.DataFrame({"subject_id": sorted(cohort_subj)})
S0["base_time"] = S0.subject_id.map(dict(zip(_b.subject_id, _b.admittime)))
S0["dec_time"] = S0.subject_id.map(_first_dec); S0["last_time"] = S0.subject_id.map(_last)
S0["event"] = S0.dec_time.notna().astype(int)
S0["duration_days"] = (S0.dec_time.where(S0.event == 1, S0.last_time) - S0.base_time).dt.total_seconds() / 86400
MODELED = set(visits.subject_id.unique())
S0["group"] = np.where(S0.subject_id.isin(MODELED), "included (modelled)",
               np.where(S0.duration_days > 0, "excluded: < 2 pre-event lab visits", "excluded: no follow-up admission"))
flow = pd.DataFrame([
    ("first admissions (all patients)", n_first),
    ("  with a liver risk factor at first admission", n_risk),
    ("  minus cirrhosis at first admission", -n_risk_cirr),
    ("  minus decompensation at first admission", -n_risk_dec),
    ("at-risk cohort", len(cohort_subj)),
    ("  minus no follow-up admission", -int((S0.group == "excluded: no follow-up admission").sum())),
    ("  minus < 2 pre-event laboratory visits", -int((S0.group == "excluded: < 2 pre-event lab visits").sum())),
    ("modelled", len(MODELED)),
    ("  decompensation events (any time)", int(surv.event.sum())),
    ("  evaluable at 1 year (event <= 365 d or follow-up >= 365 d)", len(E)),
    ("    1-year events", int(E.y1.sum())),
    ("  censored before 1 year without event", int(((BL.dur < 365) & (BL.event == 0)).sum())),
    ("  with MELD-Na, FIB-4 and stage at baseline (1-year evaluable)", len(Cm)),
], columns=["step", "n"])
flow.to_csv(REVC / "Ra_cohort_flow.csv", index=False)
print(flow.to_string(index=False))

# baseline characteristics at the cohort-entry (first) admission
_bl = _b[_b.subject_id.isin(cohort_subj)][["subject_id", "hadm_id"] + list(RISK)].merge(
    labw, on="hadm_id", how="left").merge(_pat, on="subject_id", how="left")
_bl["FIB4"] = ((_bl.anchor_age * _bl.AST) / (_bl.Platelets.clip(lower=1) * np.sqrt(_bl.ALT.clip(lower=1)))).clip(0, 100)
_m = (3.78 * np.log(_bl.Bilirubin.clip(lower=1)) + 11.2 * np.log(_bl.INR.clip(lower=1))
      + 9.57 * np.log(_bl.Creatinine.clip(1, 4)) + 6.43).clip(6, 40)
_na = _bl.Sodium.clip(125, 137)
_bl["MELD_Na"] = (_m + 1.32 * (137 - _na) - 0.033 * _m * (137 - _na)).clip(6, 40)
_bl["n_labs_observed"] = _bl[OBS].notna().sum(axis=1)
_bl["female"] = (_bl.get("gender", pd.Series(index=_bl.index, dtype=object)) == "F").astype(float)
_bl = _bl.merge(S0[["subject_id", "group", "event", "duration_days"]], on="subject_id")
_bl["n_admissions"] = _bl.subject_id.map(adm_c.groupby("subject_id").size())


def _smd(a, b, binary=False):
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()
    if binary:
        p1, p0 = a.mean(), b.mean(); d = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2)
        return (p1 - p0) / d if d > 0 else np.nan
    d = np.sqrt((a.var() + b.var()) / 2)
    return (a.mean() - b.mean()) / d if d > 0 else np.nan


rows = []
inc = _bl.group == "included (modelled)"
for var, binary in ([("anchor_age", False), ("female", True)] + [(r, True) for r in RISK]
                    + [(c, False) for c in OBS] + [("MELD_Na", False), ("FIB4", False),
                    ("n_labs_observed", False), ("n_admissions", False)]):
    r = dict(variable=var)
    for gname, gdf in _bl.groupby("group"):
        x = gdf[var]
        r[gname] = (f"{100 * x.mean():.1f}%" if binary else
                    f"{x.median():.2f} [{x.quantile(.25):.2f}, {x.quantile(.75):.2f}]")
        if not binary and var in OBS + ["MELD_Na", "FIB4"]:
            r[gname] += f" (missing {100 * x.isna().mean():.0f}%)"
    r["SMD included vs excluded"] = _smd(_bl.loc[inc, var], _bl.loc[~inc, var], binary)
    rows.append(r)
tab1 = pd.DataFrame(rows)
_n = _bl.group.value_counts()
tab1 = pd.concat([tab1, pd.DataFrame([dict(variable="n", **{g: str(int(_n[g])) for g in _n.index})])], ignore_index=True)
tab1.to_csv(REVC / "Ra_included_vs_excluded.csv", index=False)
print(tab1.to_string(index=False))
_fu = _bl[_bl.group == "excluded: < 2 pre-event lab visits"]
print(f"\nExcluded with follow-up: {len(_fu):,}; decompensation events {int(_fu.event.sum())} "
      f"({100 * _fu.event.mean():.1f}%) vs modelled {100 * surv.event.mean():.1f}%")

# missingness of each lab at the first MODELLED visit
_fv = visits.groupby("subject_id").first()
miss = (100 * _fv[OBS].isna().mean()).round(1).rename("missing % at first modelled visit")
miss.to_csv(REVC / "Ra_missing_labs_first_visit.csv")
print(miss.to_string())

                                                          step      n
                               first admissions (all patients) 223452
                   with a liver risk factor at first admission  24820
                            minus cirrhosis at first admission  -3979
                       minus decompensation at first admission   -439
                                                at-risk cohort  20402
                                  minus no follow-up admission -12395
                         minus < 2 pre-event laboratory visits  -2410
                                                      modelled   5597
                              decompensation events (any time)    301
    evaluable at 1 year (event <= 365 d or follow-up >= 365 d)   3611
                                                 1-year events     79
                          censored before 1 year without event   1986
  with MELD-Na, FIB-4 and stage at baseline (1-year evaluable)   1514
       variable    e

In [12]:
# ============================================================
# STEP Rb — fold-internal standardisation (leakage fix) and K selection
#   STEP 2 computed the robust z-score (median, IQR) on ALL visits. Here the
#   median/IQR are computed on the training folds only, and every model is refit.
# ============================================================
def fit_std(vis):
    S = {}
    for c in OBS:
        x = (np.log1p(vis[c].clip(lower=0)) if c in LOG_OBS else vis[c]).dropna().values
        if len(x) < 10: S[c] = (0.0, 1.0); continue
        med = float(np.median(x)); iqr = float(np.percentile(x, 75) - np.percentile(x, 25))
        sd = iqr / 1.349 if iqr > 1e-6 else float(np.std(x)); sd = sd if sd > 1e-6 else 1.0
        S[c] = (med, sd)
    return S


def transform(vis, S):
    v = vis.copy()
    for c in OBS:
        if c in LOG_OBS: v[c] = np.log1p(v[c].clip(lower=0))
        med, sd = S[c]; v[c] = np.clip((v[c] - med) / sd, -8, 8)
    return v


def build_seqs(vt):
    out = []
    for sid, g in vt.groupby("subject_id"):
        O = g[OBS].values.astype(float)
        if len(O) < MIN_VISITS: continue
        out.append(dict(pid=sid, O=O, fib4=g["FIB4"].values, meld=g["MELD_Na"].values,
                        admit=pd.to_datetime(g["admittime"]).values))
    return out


def fit_ordered(tr, K):
    P = fit(tr, K=K, mode="baseline", n_iter=30, seed=RANDOM_SEED)
    o = np.argsort(P["mu"] @ SEVV)
    return dict(P, mu=P["mu"][o], var=P["var"][o], pi=P["pi"][o] / P["pi"][o].sum(), A=P["A"][np.ix_(o, o)])


# sanity: the global-STD path reproduces STEP 2 exactly
_chk = build_seqs(transform(visits, fit_std(visits)))
_err = max(float(np.max(np.abs(np.nan_to_num(a["O"], nan=1e9) - np.nan_to_num(b["O"], nan=1e9))))
           for a, b in zip(_chk, seqs))
assert len(_chk) == len(seqs) and _err < 1e-9, f"re-implementation differs from STEP 2 (max diff {_err})"
print(f"STEP 2 transform reproduced (max diff {_err:.1e}).")

PIDS = np.array([s["pid"] for s in seqs])
FOLD_PIDS = [set(PIDS[_folds[f]].tolist()) for f in range(K_FOLDS)]
FITS = {}
for K in K_GRID:
    for f in range(K_FOLDS):
        t0 = time.time()
        trv = visits[~visits.subject_id.isin(FOLD_PIDS[f])]; tev = visits[visits.subject_id.isin(FOLD_PIDS[f])]
        S = fit_std(trv); trs = build_seqs(transform(trv, S)); tes = build_seqs(transform(tev, S))
        P = fit_ordered(trs, K)
        n_tr = sum(len(s["O"]) for s in trs)
        n_par = (K - 1) + K * (K - 1) + 2 * K * len(OBS)
        FITS[(K, f)] = dict(P=P, S=S, test=tes, train_ll=P["loglik"], n_train_visits=n_tr, n_par=n_par,
                            bic=-2 * P["loglik"] + n_par * np.log(n_tr),
                            test_ll=heldout_loglik(P, tes)[2], test_pred_ll=obs_pred_loglik(P, tes)[2],
                            seconds=time.time() - t0)
        print(f"  K={K} fold {f}: held-out LL/visit {FITS[(K, f)]['test_ll']:.3f}  ({time.time() - t0:.0f} s)")


def baseline_table(K):
    rows = []
    for f in range(K_FOLDS):
        P = FITS[(K, f)]["P"]
        for s in FITS[(K, f)]["test"]:
            a0 = baseline_filtered_stage(P, s["O"])
            rows.append(dict(pid=s["pid"], fold=f, base_stage_filt=float((a0 * np.arange(K)).sum()),
                             base_stage_hard=int(np.argmax(a0)), base_p_top=float(a0[-1]),
                             fib4_base=float(s["fib4"][0]) if np.isfinite(s["fib4"][0]) else np.nan,
                             meld_base=float(s["meld"][0]) if np.isfinite(s["meld"][0]) else np.nan))
    T = pd.DataFrame(rows).merge(surv[["subject_id", "event", "duration_days"]].rename(columns={"subject_id": "pid"}), on="pid")
    T["dur"] = T.duration_days.clip(lower=1)
    lab = np.where((T.event == 1) & (T.dur <= 365), 1, np.where(T.dur >= 365, 0, -1)); T["y1"] = lab
    return T


ksel = []
BLK = {}
for K in K_GRID:
    T = BLK[K] = baseline_table(K)
    Ey = T[T.y1 >= 0]
    risks = [_km(T.loc[T.base_stage_hard == k, "dur"], T.loc[T.base_stage_hard == k, "event"]) for k in range(K)]
    ns = [int((T.base_stage_hard == k).sum()) for k in range(K)]
    auc, lo, hi = _boot(Ey.y1.values, Ey.base_stage_filt.values)
    ia = ru.ipcw_auc(T.dur, T.event, T.base_stage_filt, 365)
    ksel.append(dict(K=K, heldout_LL_per_visit=np.mean([FITS[(K, f)]["test_ll"] for f in range(K_FOLDS)]),
                     heldout_pred_LL_per_visit=np.mean([FITS[(K, f)]["test_pred_ll"] for f in range(K_FOLDS)]),
                     BIC_train_mean=np.mean([FITS[(K, f)]["bic"] for f in range(K_FOLDS)]),
                     AUROC_1y=auc, AUROC_lo=lo, AUROC_hi=hi, IPCW_AUC_365=ia,
                     n_by_stage=ns, risk1y_by_stage=[round(100 * r, 1) for r in risks],
                     risk_monotone=bool(np.all(np.diff(risks) >= 0))))
KSEL = pd.DataFrame(ksel); KSEL.to_csv(REVC / "Rb_K_selection.csv", index=False)
print(KSEL.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(KSEL.K, KSEL.heldout_LL_per_visit, "o-"); ax[0].set(xlabel="K", ylabel="Held-out log-likelihood per visit")
ax[1].plot(KSEL.K, KSEL.BIC_train_mean, "o-"); ax[1].set(xlabel="K", ylabel="BIC (training folds, mean)")
plt.tight_layout(); plt.savefig(REVC / "Rb_K_selection.png", dpi=300); plt.close()

# leakage-fixed K = 3 versus the published STEP 3d numbers
BL2 = BLK[K_STAGE]
E2 = BL2[BL2.y1 >= 0].copy()
Cm2 = E2.dropna(subset=["base_stage_filt", "meld_base", "fib4_base"])
comp = []
for nm, T, EE, CC in [("published (global standardisation)", BL, E, Cm),
                      ("fold-internal standardisation", BL2, E2, Cm2)]:
    a = _boot(EE.y1.values, EE.base_stage_filt.values)
    pm = _pd(CC.y1.values, CC.base_stage_filt.values, CC.meld_base.values.astype(float))
    pf = _pd(CC.y1.values, CC.base_stage_filt.values, CC.fib4_base.values.astype(float))
    c = ru.harrell_c_boot(T.dur, T.event, T.base_stage_filt, n_boot=500)
    comp.append(dict(version=nm, n=len(T), events=int(T.event.sum()), n_1y=len(EE), events_1y=int(EE.y1.sum()),
                     AUROC_1y=f"{a[0]:.3f} ({a[1]:.3f} to {a[2]:.3f})",
                     dAUC_vs_MELDNa=f"{pm[0]:+.3f} ({pm[1]:+.3f} to {pm[2]:+.3f})",
                     dAUC_vs_FIB4=f"{pf[0]:+.3f} ({pf[1]:+.3f} to {pf[2]:+.3f})",
                     C_index_stage=f"{c[0]:.3f} ({c[1]:.3f} to {c[2]:.3f})",
                     risk1y_by_stage=[round(100 * _km(T.loc[T.base_stage_hard == k, "dur"],
                                                      T.loc[T.base_stage_hard == k, "event"]), 1) for k in range(K_STAGE)]))
COMP = pd.DataFrame(comp); COMP.to_csv(REVC / "Rb_leakage_fix_comparison.csv", index=False)
print(COMP.to_string(index=False))

# stage lab profiles (K = 3) in clinical units, median across folds, plus 1-year risk
prof = []
for f in range(K_FOLDS):
    P, S = FITS[(K_STAGE, f)]["P"], FITS[(K_STAGE, f)]["S"]
    for k in range(K_STAGE):
        r = dict(fold=f, stage=k)
        for j, c in enumerate(OBS):
            x = S[c][0] + S[c][1] * P["mu"][k, j]
            r[c] = float(np.expm1(x)) if c in LOG_OBS else float(x)
        prof.append(r)
PROF = pd.DataFrame(prof).groupby("stage")[OBS].median().round(2)
PROF["n_baseline"] = BL2.base_stage_hard.value_counts().sort_index()
PROF["1y_risk_%"] = [round(100 * _km(BL2.loc[BL2.base_stage_hard == k, "dur"], BL2.loc[BL2.base_stage_hard == k, "event"]), 1)
                     for k in range(K_STAGE)]
PROF.to_csv(REVC / "Rb_stage_lab_profiles.csv")
print("\nStage emission means in clinical units (median across fold models):")
print(PROF.to_string())

STEP 2 transform reproduced (max diff 0.0e+00).
  K=2 fold 0: held-out LL/visit -11.752  (3 s)
  K=2 fold 1: held-out LL/visit -11.576  (3 s)
  K=2 fold 2: held-out LL/visit -11.645  (3 s)
  K=2 fold 3: held-out LL/visit -11.824  (3 s)
  K=2 fold 4: held-out LL/visit -11.639  (3 s)
  K=3 fold 0: held-out LL/visit -11.206  (3 s)
  K=3 fold 1: held-out LL/visit -10.933  (3 s)
  K=3 fold 2: held-out LL/visit -11.049  (4 s)
  K=3 fold 3: held-out LL/visit -11.304  (3 s)
  K=3 fold 4: held-out LL/visit -11.077  (3 s)
  K=4 fold 0: held-out LL/visit -10.962  (3 s)
  K=4 fold 1: held-out LL/visit -10.692  (3 s)
  K=4 fold 2: held-out LL/visit -10.829  (3 s)
  K=4 fold 3: held-out LL/visit -11.049  (3 s)
  K=4 fold 4: held-out LL/visit -10.829  (3 s)
  K=5 fold 0: held-out LL/visit -10.839  (5 s)
  K=5 fold 1: held-out LL/visit -10.558  (4 s)
  K=5 fold 2: held-out LL/visit -10.708  (4 s)
  K=5 fold 3: held-out LL/visit -10.887  (4 s)
  K=5 fold 4: held-out LL/visit -10.702  (3 s)
 K  heldout_

In [13]:
# ============================================================
# STEP Rc — censoring-aware discrimination and missing-lab sensitivity
# ============================================================
rows = []
for nm, col in [("Baseline latent stage", "base_stage_filt"), ("MELD-Na (baseline)", "meld_base"),
                ("FIB-4 (baseline)", "fib4_base")]:
    d = BL2.dropna(subset=[col])
    for h in (180, 365, 730):
        a = ru.ipcw_auc_boot(d.dur, d.event, d[col], h, n_boot=500)
        rows.append(dict(score=nm, horizon_days=h, n=len(d), events_by_h=int(((d.event == 1) & (d.dur <= h)).sum()),
                         IPCW_AUC=a[0], lo=a[1], hi=a[2]))
    c = ru.harrell_c_boot(d.dur, d.event, d[col], n_boot=500)
    rows.append(dict(score=nm, horizon_days="C-index (all follow-up)", n=len(d), events_by_h=int(d.event.sum()),
                     IPCW_AUC=c[0], lo=c[1], hi=c[2]))
CENS = pd.DataFrame(rows); CENS.to_csv(REVC / "Rc_ipcw_auc_cindex.csv", index=False)
print(CENS.round(3).to_string(index=False))

# missing labs: discrimination by number of labs observed at the first modelled visit
_nobs = visits.groupby("subject_id").first()[OBS].notna().sum(axis=1).rename("n_obs")
E2 = E2.merge(_nobs, left_on="pid", right_index=True, how="left")
E2["n_obs_band"] = pd.cut(E2.n_obs, [0, 4, 7, 10], labels=["1-4 labs", "5-7 labs", "8-10 labs"])
rows = []
for band, g in E2.groupby("n_obs_band", observed=True):
    if g.y1.sum() >= 5 and (g.y1 == 0).sum() >= 5:
        a = _boot(g.y1.values, g.base_stage_filt.values)
        rows.append(dict(subset=str(band), n=len(g), events=int(g.y1.sum()), AUROC=a[0], lo=a[1], hi=a[2]))
g = E2[E2.n_obs == len(OBS)]
if g.y1.sum() >= 5:
    a = _boot(g.y1.values, g.base_stage_filt.values)
    rows.append(dict(subset="complete case (all 10 labs)", n=len(g), events=int(g.y1.sum()), AUROC=a[0], lo=a[1], hi=a[2]))
MISS = pd.DataFrame(rows); MISS.to_csv(REVC / "Rc_missing_lab_sensitivity.csv", index=False)
print(MISS.round(3).to_string(index=False))

                score            horizon_days    n  events_by_h  IPCW_AUC    lo    hi
Baseline latent stage                     180 5597           46     0.718 0.637 0.784
Baseline latent stage                     365 5597           79     0.710 0.645 0.766
Baseline latent stage                     730 5597          120     0.702 0.652 0.752
Baseline latent stage C-index (all follow-up) 5597          301     0.681 0.644 0.715
   MELD-Na (baseline)                     180 2599           29     0.667 0.567 0.778
   MELD-Na (baseline)                     365 2599           52     0.657 0.588 0.733
   MELD-Na (baseline)                     730 2599           82     0.629 0.574 0.693
   MELD-Na (baseline) C-index (all follow-up) 2599          175     0.647 0.602 0.697
     FIB-4 (baseline)                     180 3349           35     0.651 0.548 0.749
     FIB-4 (baseline)                     365 3349           60     0.641 0.561 0.712
     FIB-4 (baseline)                     730 3349    

In [14]:
import importlib; importlib.reload(ru)

<module 'revision_utils' from '/Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/revision_utils.py'>

In [15]:
# ============================================================
# STEP Rd — operating characteristics at pre-specified thresholds (1-year, evaluable patients)
#   Recalibration uses the same patient folds as the HMM (no patient in its own calibration fit).
# ============================================================
from sklearn.model_selection import PredefinedSplit
_ps = PredefinedSplit(E2.fold.values)
E2["p_stage"] = cross_val_predict(LogisticRegression(max_iter=2000), E2[["base_stage_filt"]].values,
                                  E2.y1.values, cv=_ps, method="predict_proba")[:, 1]
print(ru.wide(ru.evaluate(E2.y1, E2.p_stage, E2.pid, n_boot=2000, seed=42, label="stage, OOF recalibrated")).to_string())
rows = []
rules = [("stage >= 1 (hard)", "base_stage_hard", 1), ("stage = 2 (hard)", "base_stage_hard", 2),
         ("predicted 1-y risk >= 2%", "p_stage", 0.02), ("predicted 1-y risk >= 5%", "p_stage", 0.05),
         ("MELD-Na >= 10", "meld_base", 10), ("MELD-Na >= 15", "meld_base", 15),
         ("FIB-4 >= 1.45", "fib4_base", 1.45), ("FIB-4 >= 3.25", "fib4_base", 3.25)]
for subset, D in [("all 1-year evaluable", E2), ("common subset (all three scores)", E2.dropna(subset=["meld_base", "fib4_base"]))]:
    for nm, col, thr in rules:
        d = D.dropna(subset=[col])
        r = ru.confusion_at(d.y1, d[col], thr)
        r.update(subset=subset, rule=nm, n=len(d), events=int(d.y1.sum()),
                 flagged_per_1000=1000 * r["flagged_frac"], true_pos_per_1000=1000 * r["TP"] / len(d),
                 false_pos_per_1000=1000 * r["FP"] / len(d))
        rows.append(r)
OPC = pd.DataFrame(rows)[["subset", "rule", "n", "events", "sensitivity", "specificity", "PPV", "NPV",
                          "flagged_per_1000", "true_pos_per_1000", "false_pos_per_1000"]]
OPC.to_csv(REVC / "Rd_operating_characteristics.csv", index=False)
print(OPC.round(3).to_string(index=False))

                                          AUROC                   AUPRC                   Brier                     CITL                   Slope                     E/O
model                                                                                                                                                                   
stage, OOF recalibrated  0.706 (0.644 to 0.765)  0.048 (0.035 to 0.069)  0.021 (0.017 to 0.025)  0.000 (-0.256 to 0.204)  1.017 (0.747 to 1.312)  1.000 (0.822 to 1.280)
                          subset                     rule    n  events  sensitivity  specificity   PPV   NPV  flagged_per_1000  true_pos_per_1000  false_pos_per_1000
            all 1-year evaluable        stage >= 1 (hard) 3611      79        0.684        0.712 0.050 0.990           296.871             14.954             281.916
            all 1-year evaluable         stage = 2 (hard) 3611      79        0.418        0.817 0.049 0.984           188.313              9.139             179

In [16]:
# ============================================================
# STEP Re — supervised baselines on the same folds + static-mixture ablation
# ============================================================
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
try:
    import xgboost as xgb; _HX = True
except Exception:
    _HX = False
_fvis = visits.groupby("subject_id").first()
XB = _fvis[OBS + ["anchor_age"]].reindex(E2.pid).reset_index(drop=True)
XB["MELD_Na"] = E2.meld_base.values; XB["FIB4"] = E2.fib4_base.values
base_models = {
    "Logistic (10 labs + age + MELD-Na + FIB-4)": make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                                    StandardScaler(), LogisticRegression(max_iter=3000)),
    "Random forest": make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                   RandomForestClassifier(n_estimators=500, min_samples_leaf=10, n_jobs=-1, random_state=RANDOM_SEED)),
    ("XGBoost" if _HX else "Gradient boosting"): (xgb.XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                                  subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED,
                                                  eval_metric="logloss") if _HX else
                                                  HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05,
                                                  max_depth=3, early_stopping=False, random_state=RANDOM_SEED)),
}
rows, dl = [ru.evaluate(E2.y1, E2.p_stage, E2.pid, n_boot=2000, seed=42, label="Latent stage (label-free, 1-parameter recalibration)",
                        rank_score=E2.base_stage_filt)], []
for nm, mdl in base_models.items():
    p = cross_val_predict(mdl, XB.values.astype(float), E2.y1.values, cv=_ps, method="predict_proba")[:, 1]
    rows.append(ru.evaluate(E2.y1, p, E2.pid, n_boot=2000, seed=42, label=nm))
    d = ru.paired_delta_auc(E2.y1.values, p, E2.base_stage_filt.values, E2.pid.values, n_boot=2000, seed=42)
    dl.append(dict(model=nm, delta_auroc_vs_stage=d[0], lo=d[1], hi=d[2]))

# static mixture ablation: same emissions, trained on visits as independent observations (no dynamics)
st_rows = []
for f in range(K_FOLDS):
    trv = visits[~visits.subject_id.isin(FOLD_PIDS[f])]
    S = FITS[(K_STAGE, f)]["S"]
    _t = transform(trv, S)
    singles = [dict(O=r[None, :]) for r in _t[OBS].values.astype(float) if np.isfinite(r).any()]
    Pm = fit_ordered(singles, K_STAGE)
    for s in FITS[(K_STAGE, f)]["test"]:
        a0 = baseline_filtered_stage(Pm, s["O"])
        st_rows.append(dict(pid=s["pid"], static_stage=float((a0 * np.arange(K_STAGE)).sum())))
    print(f"  static mixture fold {f} fitted")
_st = pd.DataFrame(st_rows).set_index("pid").static_stage
E2["static_stage"] = E2.pid.map(_st)
a = _boot(E2.y1.values, E2.static_stage.values)
d = _pd(E2.y1.values, E2.base_stage_filt.values, E2.static_stage.values)
dl.append(dict(model="ablation: static mixture (no temporal structure in training)", delta_auroc_vs_stage=-d[0],
               lo=-d[2], hi=-d[1]))
print(f"static-mixture stage AUROC {a[0]:.3f} ({a[1]:.3f} to {a[2]:.3f}); HMM-trained minus static = {d[0]:+.3f} ({d[1]:+.3f} to {d[2]:+.3f})")
BASE = pd.concat(rows, ignore_index=True); BASED = pd.DataFrame(dl)
BASE.to_csv(REVC / "Re_baselines.csv", index=False); BASED.to_csv(REVC / "Re_baselines_delta.csv", index=False)
print(ru.wide(BASE).to_string()); print(BASED.round(3).to_string(index=False))

  static mixture fold 0 fitted
  static mixture fold 1 fitted
  static mixture fold 2 fitted
  static mixture fold 3 fitted
  static mixture fold 4 fitted
static-mixture stage AUROC 0.711 (0.649 to 0.770); HMM-trained minus static = +0.001 (-0.004 to +0.007)
                                                                       AUROC                   AUPRC                   Brier                      CITL                   Slope                     E/O
model                                                                                                                                                                                                 
Latent stage (label-free, 1-parameter recalibration)  0.711 (0.650 to 0.771)  0.060 (0.039 to 0.093)  0.021 (0.017 to 0.025)   0.000 (-0.256 to 0.204)  1.017 (0.747 to 1.312)  1.000 (0.822 to 1.280)
Logistic (10 labs + age + MELD-Na + FIB-4)            0.705 (0.646 to 0.763)  0.048 (0.035 to 0.070)  0.022 (0.018 to 0.027)  -0.040 (-0.312 to 

In [17]:
# ============================================================
# STEP Rf — stability of the learned stages and attribution fidelity
# ============================================================
# (a) fold-refit agreement: every fold model assigns a baseline stage to EVERY patient
_first = {s["pid"]: s for s in seqs}
_vis_first = visits.groupby("subject_id").head(1)
H = np.zeros((len(PIDS), K_FOLDS), int)
for f in range(K_FOLDS):
    P, S = FITS[(K_STAGE, f)]["P"], FITS[(K_STAGE, f)]["S"]
    O1 = transform(_vis_first.set_index("subject_id").loc[PIDS].reset_index(), S)[OBS].values.astype(float)
    for i in range(len(PIDS)):
        H[i, f] = int(np.argmax(baseline_filtered_stage(P, O1[i:i + 1])))
kap = [cohen_kappa_score(H[:, a], H[:, b]) for a in range(K_FOLDS) for b in range(a + 1, K_FOLDS)]
agree = [(H[:, a] == H[:, b]).mean() for a in range(K_FOLDS) for b in range(a + 1, K_FOLDS)]
mu_sd = np.std(np.stack([FITS[(K_STAGE, f)]["P"]["mu"] for f in range(K_FOLDS)]), axis=0).mean()
# (b) bootstrap refits of fold-0 training patients, agreement on fold-0 test patients
N_REFIT = int(os.environ.get("REV_REFIT", "3"))
trv0 = visits[~visits.subject_id.isin(FOLD_PIDS[0])]; S0f = FITS[(K_STAGE, 0)]["S"]
tr_seqs0 = build_seqs(transform(trv0, S0f)); te0 = FITS[(K_STAGE, 0)]["test"]
ref = np.array([np.argmax(baseline_filtered_stage(FITS[(K_STAGE, 0)]["P"], s["O"])) for s in te0])
bk = []
for b in range(N_REFIT):
    rb = np.random.default_rng(100 + b).integers(0, len(tr_seqs0), len(tr_seqs0))
    Pb = fit_ordered([tr_seqs0[i] for i in rb], K_STAGE)
    hb = np.array([np.argmax(baseline_filtered_stage(Pb, s["O"])) for s in te0])
    bk.append(cohen_kappa_score(ref, hb)); print(f"  bootstrap refit {b}: kappa {bk[-1]:.3f}")
STAB = pd.DataFrame([
    dict(check="fold-refit pairwise Cohen kappa, mean (min)", value=f"{np.mean(kap):.3f} ({np.min(kap):.3f})"),
    dict(check="fold-refit pairwise exact agreement, mean (min)", value=f"{np.mean(agree):.3f} ({np.min(agree):.3f})"),
    dict(check="SD of stage emission means across folds (z units, mean over stages x labs)", value=f"{mu_sd:.3f}"),
    dict(check=f"bootstrap-refit kappa vs original on fold-0 test (n={N_REFIT})", value=f"{np.mean(bk):.3f} (min {np.min(bk):.3f})"),
])

# (c) fidelity: exact log-odds identity, deletion test, noise stability (top stage vs stage 0)
rng = np.random.default_rng(RANDOM_SEED)
rec = []
for f in range(K_FOLDS):
    P = FITS[(K_STAGE, f)]["P"]; mu, var, pi = P["mu"], P["var"], P["pi"]
    for s in FITS[(K_STAGE, f)]["test"]:
        o = s["O"][0]; ob = ~np.isnan(o)
        if ob.sum() < 2: continue
        llf = lambda x, k: -0.5 * (np.log(2 * np.pi * var[k]) + (x - mu[k]) ** 2 / var[k])
        LLR = np.where(ob, llf(o, K_STAGE - 1) - llf(o, 0), np.nan)
        NRM = np.where(ob, (mu[-1] - mu[0]) * (o - 0.5 * (mu[-1] + mu[0])), np.nan)   # variance-normalised (STEP 4)
        a0 = baseline_filtered_stage(P, o[None, :])
        ident = abs(np.log(a0[-1] / a0[0]) - (np.log(pi[-1] / pi[0]) + np.nansum(LLR)))
        es = (a0 * np.arange(K_STAGE)).sum(); dele = np.full(len(OBS), np.nan)
        for j in np.nonzero(ob)[0]:
            oo = o.copy(); oo[j] = np.nan
            dele[j] = es - (baseline_filtered_stage(P, oo[None, :]) * np.arange(K_STAGE)).sum()
        on = o + rng.normal(0, 0.05, len(o))
        LLRn = np.where(ob, llf(on, K_STAGE - 1) - llf(on, 0), np.nan)
        rec.append(dict(ident=ident, top_llr=np.nanargmax(LLR), top_nrm=np.nanargmax(NRM), top_del=np.nanargmax(dele),
                        top_noise=np.nanargmax(LLRn), nobs=int(ob.sum()),
                        rho=pd.Series(LLR[ob]).corr(pd.Series(dele[ob]), method="spearman") if ob.sum() >= 3 else np.nan,
                        sign=np.mean(np.sign(LLR[ob]) == np.sign(dele[ob]))))
R = pd.DataFrame(rec)
FID = pd.DataFrame([
    dict(check="max |log-odds identity error|", value=f"{R.ident.max():.1e}"),
    dict(check="patients (first visit, >= 2 labs)", value=str(len(R))),
    dict(check="top raw-LLR lab = lab whose deletion lowers expected stage most", value=f"{(R.top_llr == R.top_del).mean():.3f}"),
    dict(check="top normalised-LLR lab = same", value=f"{(R.top_nrm == R.top_del).mean():.3f}"),
    dict(check="median Spearman(LLR, deletion effect), >= 3 labs", value=f"{R.rho.median():.3f}"),
    dict(check="mean sign agreement LLR vs deletion effect", value=f"{R.sign.mean():.3f}"),
    dict(check="top raw-LLR lab unchanged under 0.05 SD noise", value=f"{(R.top_noise == R.top_llr).mean():.3f}"),
])
pd.concat([STAB, FID]).to_csv(REVC / "Rf_stability_fidelity.csv", index=False)
print(STAB.to_string(index=False)); print(FID.to_string(index=False))

  bootstrap refit 0: kappa 0.911
  bootstrap refit 1: kappa 0.951
  bootstrap refit 2: kappa 0.526
                                                                     check             value
                               fold-refit pairwise Cohen kappa, mean (min)     0.915 (0.842)
                           fold-refit pairwise exact agreement, mean (min)     0.959 (0.924)
SD of stage emission means across folds (z units, mean over stages x labs)             0.056
                    bootstrap-refit kappa vs original on fold-0 test (n=3) 0.796 (min 0.526)
                                                          check   value
                                  max |log-odds identity error| 4.3e-14
                              patients (first visit, >= 2 labs)    5572
top raw-LLR lab = lab whose deletion lowers expected stage most   0.882
                                  top normalised-LLR lab = same   0.654
               median Spearman(LLR, deletion effect), >= 3 labs   0.943
    

In [18]:
# ============================================================
# STEP Rg — temporal validation by MIMIC-IV anchor_year_group
# ============================================================
if "anchor_year_group" in _pat.columns:
    AYG = _pat.set_index("subject_id")["anchor_year_group"].reindex(PIDS)
    DEV_G, TEST_G = ru.temporal_split(AYG, 0.20)
    dev_p = set(AYG[AYG.isin(DEV_G)].index); tt_p = set(AYG[AYG.isin(TEST_G)].index)
    trv = visits[visits.subject_id.isin(dev_p)]; ttv = visits[visits.subject_id.isin(tt_p)]
    S = fit_std(trv); Pd = fit_ordered(build_seqs(transform(trv, S)), K_STAGE)
    rows = []
    for s in build_seqs(transform(ttv, S)):
        a0 = baseline_filtered_stage(Pd, s["O"])
        rows.append(dict(pid=s["pid"], base_stage_filt=float((a0 * np.arange(K_STAGE)).sum()),
                         meld_base=s["meld"][0], fib4_base=s["fib4"][0]))
    TV = pd.DataFrame(rows).merge(surv[["subject_id", "event", "duration_days"]].rename(columns={"subject_id": "pid"}), on="pid")
    TV["dur"] = TV.duration_days.clip(lower=1)
    TV["y1"] = np.where((TV.event == 1) & (TV.dur <= 365), 1, np.where(TV.dur >= 365, 0, -1))
    TE = TV[TV.y1 >= 0]
    out_rows = []
    for nm, col in [("Latent stage (fit on development years)", "base_stage_filt"), ("MELD-Na", "meld_base"), ("FIB-4", "fib4_base")]:
        d = TE.dropna(subset=[col]); dd = TV.dropna(subset=[col])
        a = _boot(d.y1.values, d[col].values) if d.y1.sum() >= 5 else (np.nan,) * 3
        ia = ru.ipcw_auc_boot(dd.dur, dd.event, dd[col], 365, n_boot=500)
        out_rows.append(dict(score=nm, dev_groups="; ".join(DEV_G), test_groups="; ".join(TEST_G),
                             n_test=len(dd), events_1y=int(d.y1.sum()), AUROC_1y=a[0], lo=a[1], hi=a[2],
                             IPCW_AUC_365=ia[0], ipcw_lo=ia[1], ipcw_hi=ia[2]))
    TVAL = pd.DataFrame(out_rows); TVAL.to_csv(REVC / "Rg_temporal_validation.csv", index=False)
    print(AYG.value_counts().sort_index().to_string())
    print(TVAL.round(3).to_string(index=False))
else:
    print("anchor_year_group not found in patients.csv; temporal validation skipped.")
print("\nRevision outputs written to", REVC.resolve())

anchor_year_group
2008 - 2010    1967
2011 - 2013    1284
2014 - 2016    1121
2017 - 2019     801
2020 - 2022     424
                                  score                            dev_groups              test_groups  n_test  events_1y  AUROC_1y    lo    hi  IPCW_AUC_365  ipcw_lo  ipcw_hi
Latent stage (fit on development years) 2008 - 2010; 2011 - 2013; 2014 - 2016 2017 - 2019; 2020 - 2022    1225         23     0.613 0.485 0.730         0.612    0.478    0.719
                                MELD-Na 2008 - 2010; 2011 - 2013; 2014 - 2016 2017 - 2019; 2020 - 2022     711         17     0.503 0.375 0.637         0.498    0.353    0.636
                                  FIB-4 2008 - 2010; 2011 - 2013; 2014 - 2016 2017 - 2019; 2020 - 2022     865         18     0.556 0.423 0.695         0.558    0.411    0.693

Revision outputs written to /Users/sumiyasayeed/Downloads/MSc/PEMDT - Diseases/JBI/PEMDT-cirrhosis/results/progression_output/revision


In [19]:
# ============================================================
# STEP 3b — 1-year decompensation calibration (5-fold OOF) + Brier
# ============================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import brier_score_loss

_cc = [c for c in ["peak_stage", "stage_velocity", "dwell_top", "stage_entropy",
                   "meld_max", "fib4_max"] if c in Dd.columns]
_Xc = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Dd[_cc]))
# plain logistic (NO class_weight) so predicted probabilities are calibratable
_p = cross_val_predict(LogisticRegression(max_iter=2000), _Xc, y, cv=5,
                       method="predict_proba")[:, 1]
_brier = brier_score_loss(y, _p)
print(f"=== 1-year decompensation calibration (5-fold OOF, Brier = {_brier:.3f}) ===")
_edges = np.quantile(_p, np.linspace(0, 1, 6)); _edges[0] -= 1e-9
_bin = np.digitize(_p, _edges[1:-1])
for b in range(5):
    m = _bin == b
    if m.sum() < 10: continue
    print(f"  bin {b}: n={int(m.sum()):<4} predicted={_p[m].mean():.3f}  observed={y[m].mean():.3f}")

=== 1-year decompensation calibration (5-fold OOF, Brier = 0.021) ===
  bin 0: n=212  predicted=0.006  observed=0.024
  bin 1: n=211  predicted=0.011  observed=0.005
  bin 2: n=211  predicted=0.012  observed=0.009
  bin 3: n=211  predicted=0.017  observed=0.024
  bin 4: n=212  predicted=0.059  observed=0.047


## Step 4 — Explainability (normalized LLR vs SHAP)

In [20]:
# ============================================================
# EXPLAINABILITY — LLR (raw + variance-normalized) vs SHAP
# ============================================================
from scipy.stats import spearmanr
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
try:
    import shap as _shap; _HS = True
except Exception:
    _HS = False; print("shap not installed -> exact-linear fallback")

_P = params; _S = seqs; _OBS = list(OBS); _K = K_STAGE
_mu, _var = _P["mu"], _P["var"]

def _llr_feat(o, tgt, ref=0, unit=False):
    out = {}
    for j, f in enumerate(_OBS):
        if np.isnan(o[j]): continue
        if unit:   # unit-variance / LDA-style: scale-invariant on standardized obs
            out[f] = (_mu[tgt,j]-_mu[ref,j]) * (o[j]-0.5*(_mu[tgt,j]+_mu[ref,j]))
        else:      # original variance-weighted LLR
            lt = -0.5*(np.log(2*np.pi*_var[tgt,j]) + (o[j]-_mu[tgt,j])**2/_var[tgt,j])
            lr = -0.5*(np.log(2*np.pi*_var[ref,j]) + (o[j]-_mu[ref,j])**2/_var[ref,j])
            out[f] = lt - lr
    return out

_raw = {f: [] for f in _OBS}; _nrm = {f: [] for f in _OBS}; _X = []; _Y = []
for s in _S:
    g, st, _ = posterior_stage(_P, s["O"]); hard = np.argmax(g, axis=1)
    for t in range(len(st)):
        _X.append(s["O"][t]); _Y.append(int(hard[t] == _K-1))
        if hard[t] == _K-1:
            for f, v in _llr_feat(s["O"][t], _K-1, 0, False).items(): _raw[f].append(abs(v))
            for f, v in _llr_feat(s["O"][t], _K-1, 0, True ).items(): _nrm[f].append(abs(v))
Xv = pd.DataFrame(_X, columns=_OBS); Xv = Xv.fillna(Xv.median()); Yv = np.array(_Y)
print(f"visit-level: {len(Xv):,} visits | advanced (top stage) = {int(Yv.sum()):,} ({Yv.mean()*100:.1f}%)")

llr_raw  = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _raw.items()})
llr_norm = pd.Series({f:(np.mean(v) if v else 0.0) for f,v in _nrm.items()})
_gbm = GradientBoostingClassifier(max_depth=3, n_estimators=200, random_state=0).fit(Xv, Yv)
if _HS:
    _r = _shap.TreeExplainer(_gbm).shap_values(Xv)
    if isinstance(_r, list): _r = _r[-1]
    _r = np.asarray(_r)
    if _r.ndim == 3: _r = _r[:, :, -1]
    shap_imp = pd.Series(np.abs(_r).mean(0), index=_OBS)
else:
    _lin = LogisticRegression(max_iter=2000).fit(Xv, Yv)
    shap_imp = pd.Series(np.abs(_lin.coef_[0])*Xv.std().values, index=_OBS)

cmp = pd.DataFrame({"LLR_raw":llr_raw, "LLR_norm":llr_norm, "SHAP":shap_imp})
for c in ["LLR_raw","LLR_norm","SHAP"]: cmp[c+"_rk"] = cmp[c].rank(ascending=False)
cmp = cmp.sort_values("SHAP", ascending=False)
rho_raw  = spearmanr(cmp["LLR_raw_rk"],  cmp["SHAP_rk"]).correlation
rho_norm = spearmanr(cmp["LLR_norm_rk"], cmp["SHAP_rk"]).correlation
print("\n=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===")
print(cmp[["LLR_raw","LLR_norm","SHAP","LLR_raw_rk","LLR_norm_rk","SHAP_rk"]].round(3).to_string())
print(f"\nSpearman  SHAP vs raw-LLR  = {rho_raw:+.3f}")
print(f"Spearman  SHAP vs norm-LLR = {rho_norm:+.3f}")
print("  If norm-LLR agrees with SHAP much better, the raw-LLR ranking was a variance artifact.")


visit-level: 23,636 visits | advanced (top stage) = 4,479 (18.9%)

=== Feature importance: raw-LLR vs norm-LLR vs SHAP (advanced stage) ===
            LLR_raw  LLR_norm   SHAP  LLR_raw_rk  LLR_norm_rk  SHAP_rk
INR          16.339     5.459  0.845         1.0          1.0      1.0
Bilirubin     6.771     3.384  0.693         2.0          2.0      2.0
AST           1.930     1.279  0.492         4.0          3.0      3.0
ALT           1.544     0.996  0.428         5.0          4.0      4.0
Creatinine    2.520     0.640  0.372         3.0          6.0      5.0
Platelets     1.106     0.098  0.269         6.0          9.0      6.0
Sodium        0.635     0.457  0.267         8.0          7.0      7.0
Hemoglobin    0.555     0.349  0.164         9.0          8.0      8.0
Albumin       1.105     0.848  0.141         7.0          5.0      9.0
WBC           0.387     0.069  0.095        10.0         10.0     10.0

Spearman  SHAP vs raw-LLR  = +0.927
Spearman  SHAP vs norm-LLR = +0.842
  If n

## Step 5 — Visualization
Figures saved to `results/progression_output/figures/`.

In [21]:
# ============================================================
# VIZ setup + shared palette + V1: KM by baseline stage
# ============================================================
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)

# unified blue-primary scheme (matches PEMDT-AKI and respiratory)
RED, BLUE, GREEN, GREY, AMBER = "#e63946", "#1f77b4", "#2ca02c", "#9aa0a6", "#f4a261"
STAGE_COLORS = [BLUE, GREEN, AMBER, RED]  # stages 0->3 (blue, green, amber, red only worst)
C_STAGE = BLUE    # latent stage / PEMDT model (primary)
C_MELD  = GREEN   # MELD-Na
C_FIB   = AMBER   # FIB-4
C_SHAP  = GREEN   # SHAP series
C_REF   = GREY    # reference / chance / perfect line (neutral, not a data color)

# --- evaluation figures use the OUT-OF-FOLD pooled set if available (powered + leakage-free) ---
Dfig  = Do if "Do" in globals() else D
Ddfig = Dq if "Dq" in globals() else Dd
yfig  = yo if "yo" in globals() else y
print("figures using:", "OOF pooled set" if "Do" in globals() else "SF_test (run the OOF cell for pooled figures)")

def km_curve(dur, ev, tmax=730):
    dur = np.asarray(dur, float); ev = np.asarray(ev, int)
    ts = np.unique(dur[ev == 1]); ts = ts[ts <= tmax]
    surv = 1.0; xs = [0.0]; ys = [0.0]
    for ut in ts:
        d = int(((dur == ut) & (ev == 1)).sum()); r = int((dur >= ut).sum())
        if r > 0: surv *= (1 - d / r)
        xs.append(ut); ys.append(1 - surv)
    xs.append(tmax); ys.append(ys[-1])
    return np.array(xs), np.array(ys) * 100

plt.figure(figsize=(7, 5))
for k in range(K_STAGE):
    m = Dfig["base_stage"].round() == k
    if m.sum() < 20: continue
    x, yv = km_curve(Dfig.loc[m, "dur"], Dfig.loc[m, "event"])
    r1 = yv[np.searchsorted(x, 365, "right") - 1]
    plt.step(x / 365, yv, where="post", lw=2.2, color=STAGE_COLORS[k % 4],
             label=f"Baseline stage {k} (n={int(m.sum())}, 1-yr {r1:.1f}%)")
plt.xlabel("Years from baseline"); plt.ylabel("Cumulative decompensation risk (%)")
plt.title("Decompensation by baseline latent stage")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(FIG / "km_by_stage.png", dpi=150); plt.show()


figures using: OOF pooled set


In [22]:
# ============================================================
# V2: ROC — latent stage vs MELD-Na vs FIB-4 (1-year discrimination)
# ============================================================
from sklearn.metrics import roc_curve, roc_auc_score
plt.figure(figsize=(6, 6))
for name, col, c in [("Latent peak stage", "peak_stage", C_STAGE),
                     ("MELD-Na", "meld_max", C_MELD),
                     ("FIB-4", "fib4_max", C_FIB)]:
    if col not in Ddfig.columns: continue
    m = Ddfig[col].notna().values
    if len(np.unique(yfig[m])) < 2: continue
    s = Ddfig[col].values[m]
    fpr, tpr, _ = roc_curve(yfig[m], s)
    plt.plot(fpr, tpr, lw=2, color=c, label=f"{name} (AUC={roc_auc_score(yfig[m], s):.3f})")
plt.plot([0, 1], [0, 1], "--", color=C_REF, lw=1)
plt.xlabel("False-positive rate"); plt.ylabel("True-positive rate")
plt.title("1-year decompensation discrimination")
plt.legend(loc="lower right"); plt.tight_layout()
plt.savefig(FIG / "roc_stage_vs_scores.png", dpi=150); plt.show()


In [23]:
# ============================================================
# V3: Calibration curve (predicted vs observed 1-year risk)
# ============================================================
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import brier_score_loss
_cc = [c for c in ["peak_stage", "stage_velocity", "dwell_top", "stage_entropy",
                   "meld_max", "fib4_max"] if c in Ddfig.columns]
_X = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(Ddfig[_cc]))
p = cross_val_predict(LogisticRegression(max_iter=2000), _X, yfig, cv=5, method="predict_proba")[:, 1]
edges = np.quantile(p, np.linspace(0, 1, 6)); edges[0] -= 1e-9; b = np.digitize(p, edges[1:-1])
xs = [p[b == i].mean() for i in range(5) if (b == i).sum()]
ys = [yfig[b == i].mean() for i in range(5) if (b == i).sum()]
hi = max(max(xs), max(ys)) * 1.1
plt.figure(figsize=(6, 6))
plt.plot([0, hi], [0, hi], "--", color=C_REF, label="perfect")
plt.plot(xs, ys, "o-", color=C_STAGE, lw=2, label="model")
plt.xlabel("Predicted 1-year risk"); plt.ylabel("Observed 1-year risk")
plt.title(f"1-year risk calibration (Brier={brier_score_loss(yfig, p):.3f})")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(FIG / "calibration_1yr.png", dpi=150); plt.show()


In [24]:
# ============================================================
# V4: Cox hazard-ratio forest plot (colored by feature group)
# ============================================================
if HAVE_LL:
    def _hr(cols):
        cc = [c for c in cols if c in Dfig.columns]; d = Dfig[cc + ["dur", "event"]].dropna()
        for c in cc: d[c] = (d[c] - d[c].mean()) / (d[c].std() + 1e-9)
        cph = CoxPHFitter(penalizer=0.1).fit(d, "dur", "event"); h = cph.summary
        return [(c, h.loc[c, "exp(coef)"], h.loc[c, "exp(coef) lower 95%"],
                 h.loc[c, "exp(coef) upper 95%"]) for c in cc]
    rows = _hr(["peak_stage", "stage_velocity", "dwell_top", "stage_entropy"]) + \
           _hr(["meld_max", "fib4_max"])
    _cmap = {"meld_max": C_MELD, "fib4_max": C_FIB}
    lab = [r[0] for r in rows]; hr = np.array([r[1] for r in rows])
    lo = np.array([r[2] for r in rows]); hiv = np.array([r[3] for r in rows])
    yv = np.arange(len(rows))[::-1]
    plt.figure(figsize=(7, 4))
    for i in range(len(rows)):
        plt.errorbar(hr[i], yv[i], xerr=[[hr[i] - lo[i]], [hiv[i] - hr[i]]],
                     fmt="o", color=_cmap.get(lab[i], C_STAGE), capsize=3)
    plt.axvline(1, color=C_REF, ls="--"); plt.yticks(yv, lab)
    plt.xlabel("Hazard ratio per 1 SD (95% CI)")
    plt.title("Cox proportional hazards"); plt.tight_layout()
    plt.savefig(FIG / "cox_forest.png", dpi=150); plt.show()
else:
    print("lifelines not available; skipping forest plot")


In [25]:
# ============================================================
# V5: Example patient latent-stage trajectory (digital-twin view)
# ============================================================
_prog = set(surv.loc[surv.event == 1, "subject_id"])
_cand = [s for s in seqs if len(s["O"]) >= 5 and s["pid"] in _prog] or \
        [s for s in seqs if len(s["O"]) >= 5]
s = max(_cand, key=lambda z: len(z["O"]))
g, st, _ = posterior_stage(params, s["O"])
days = (pd.to_datetime(s["admit"]).astype("int64") / 1e9 / 86400); days = days - days[0]
plt.figure(figsize=(9, 4))
plt.stackplot(days, *[g[:, k] for k in range(K_STAGE)],
              labels=[f"stage {k}" for k in range(K_STAGE)],
              colors=STAGE_COLORS[:K_STAGE], alpha=.85)
plt.plot(days, st / (K_STAGE - 1), "k--", lw=1.3, label="hard stage / max")
plt.xlabel("Days from first visit"); plt.ylabel("Posterior belief"); plt.ylim(0, 1)
plt.title(f"Latent-stage trajectory (patient {s['pid']})")
plt.legend(ncol=3, fontsize=8, loc="upper left"); plt.tight_layout()
plt.savefig(FIG / "patient_trajectory.png", dpi=150); plt.show()


In [26]:
# ============================================================
# V6: LLR vs SHAP feature importance (needs `cmp` from explainability cell)
# ============================================================
try:
    _c = cmp.copy()
    _lcol = "LLR_norm" if "LLR_norm" in _c.columns else ("LLR" if "LLR" in _c.columns else "LLR_raw")
    _c = _c.sort_values("SHAP", ascending=False)
    def _nz(sr): return sr / (sr.max() if sr.max() > 0 else 1.0)
    xv = np.arange(len(_c)); w = 0.4
    plt.figure(figsize=(9, 4))
    plt.bar(xv - w / 2, _nz(_c[_lcol]), w, label="LLR (normalized)", color=C_STAGE)
    plt.bar(xv + w / 2, _nz(_c["SHAP"]), w, label="SHAP", color=C_SHAP)
    plt.xticks(xv, _c.index, rotation=45, ha="right"); plt.ylabel("Relative importance")
    plt.title("Feature importance: LLR vs SHAP (advanced stage)")
    plt.legend(); plt.tight_layout()
    plt.savefig(FIG / "llr_vs_shap.png", dpi=150); plt.show()
except NameError:
    print("Run the explainability cell (defines `cmp`) first.")
